# 📚 Guía de estudio — Control de Python
## Transformación Digital · Series de tiempo
### Basada en las clases de la profesora Beatriz Berríos P.

Este notebook junta **las 3 clases de Python** en un solo archivo, en orden, para que estudies ejecutando celda por celda.

**Cómo leer este archivo**

| Lo que ves | Quién lo escribió | Para qué |
|---|---|---|
| Celdas de código (sin emoji) y los títulos de cada clase | **La profe, copiado tal cual** de sus notebooks | Es lo que vas a copiar en el control |
| 📝 **Apunte** | Yo | Explico qué hace el código, línea por línea |
| 📊 **Cómo leer el gráfico / la tabla** | Yo | Qué mirar y cómo interpretarlo |
| 🔁 **Si en el control me cambian…** | Yo | Qué líneas tienes que modificar si cambian los datos o los números |
| ⚠️ **Ojo** | Yo | Trampas típicas y errores que te pueden aparecer |
| ✅ **Autoevaluación** | Yo | Preguntas para practicar (haz clic en “Ver respuesta”) |

> No agregué código nuevo a los ejercicios: todos los códigos son los de la profe. Cuando te digo “🧪 Prueba”, es para que cambies **un número** en una celda de la profe y la vuelvas a correr.

**Antes de empezar** (solo si te falta algún paquete), ejecuta en una celda:

```python
%pip install pandas numpy matplotlib requests statsmodels scipy jinja2
```

(`jinja2` lo pide la tabla con estilos de la Clase 1, `df.style`.)

## 🗺️ El mapa del ramo en una sola idea

Las tres clases son **un mismo camino** para trabajar una serie de tiempo (datos ordenados en el tiempo, por ejemplo ventas mensuales o el dólar mensual):

1. **Clase 1 — Transformaciones:** *miro* la serie y, si tiene un problema (asimetría, varianza que cambia, atípicos, escalas distintas), la *transformo*.
2. **Clase 2 — Descomposición clásica:** separo la serie en **tendencia + estacionalidad + residuo** para entender de dónde viene cada movimiento.
3. **Clase 3 — Suavizamiento exponencial (SES, Holt, Holt-Winters):** *pronostico* el futuro, comparo modelos con datos que el modelo no vio (**fuera de muestra**) y elijo el que tiene menor **RMSE**.

Las tres comparten la misma idea de fondo:

$$\text{Serie} = \text{Tendencia} + \text{Estacionalidad} + \text{Ruido}$$

- En la **Clase 1** la profe *fabrica* una serie justamente así (100 + 1,4·t + estacionalidad + ruido).
- En la **Clase 2** la *desarma* en esas piezas.
- En la **Clase 3** cada modelo *agrega una pieza más*: SES = solo nivel; Holt = nivel + tendencia; Holt-Winters = nivel + tendencia + estacionalidad.

## 🐍 Python mínimo que necesitas para entender los códigos

| Expresión | Qué significa |
|---|---|
| `import pandas as pd` | Carga la librería y le pone un apodo (`pd`). Por eso después se escribe `pd.DataFrame`, `pd.Series`, etc. |
| `pd.DataFrame({...})` | Una **tabla** (como una hoja de Excel). Cada clave del diccionario es una columna. |
| `pd.Series(valores, index=fechas)` | Una **columna sola** con un índice; en series de tiempo el índice son fechas. |
| `df['Ventas']` | Saca la columna `Ventas` de la tabla `df`. |
| `df['Nueva'] = ...` | Crea (o reemplaza) una columna. |
| `pd.date_range('2021-01-01', periods=60, freq='MS')` | 60 fechas mensuales. `MS` = *Month Start* (día 1 de cada mes). |
| `serie.iloc[:-24]` | Todas las filas **menos las últimas 24** (posición). |
| `serie.iloc[-24:]` | Solo **las últimas 24** filas. |
| `.head()` / `.tail()` | Muestra las primeras / últimas 5 filas. |
| `.round(2)` | Redondea a 2 decimales (solo para mostrar). |
| `np.mean`, `np.abs`, `np.sqrt`, `np.log` | Promedio, valor absoluto, raíz cuadrada, logaritmo natural. |
| `np.asarray(x)` | Convierte a un arreglo simple de números (sin índice). |
| `a, b = funcion(...)` | La función devuelve **dos cosas** y se guardan en `a` y `b`. |
| `plt.figure(figsize=(ancho, alto))` … `plt.show()` | Abre un gráfico, lo configura y lo muestra. |
| `# texto` | Comentario: Python lo ignora, es solo explicación. |

⚠️ **Ojo general:** un notebook se ejecuta **en orden**. Si te sale `NameError: name 'xxx' is not defined`, casi siempre es porque no ejecutaste una celda anterior. Solución: *Run All* (Ejecutar todo) o ejecutar desde arriba.

---
# 🟦 CLASE 1 — Transformaciones de una serie
*Archivo de la profe: `transformaciones_supermercado_INE_CCS_1_1.ipynb`*

# Transformaciones de una serie de precios/ventas de supermercado
## Profesor: Beatriz Berríos P.


Ejemplo reproducible para Jupyter Notebook. Se usa una serie simulada con comportamiento similar a una serie mensual de precios o ventas. Para un trabajo formal, se puede reemplazar por datos oficiales del INE o de la CCS.

Se aplican las transformaciones indicadas: **logaritmo, Box-Cox, raíz cuadrada, Yeo-Johnson, estandarización Z y winsorización**, explicando su propósito, ventajas y desventajas.

📝 **Apunte — ¿qué es “transformar”?**

Transformar es aplicar una función matemática a **cada valor** de la serie (por ejemplo, sacarle logaritmo) para corregir un problema estadístico. **No cambia los datos “reales”**: cambia la escala en que los miramos. La idea clave de la profe (la conclusión final de su notebook) es: *se transforma para resolver un problema concreto, no por costumbre*.

La siguiente celda **fabrica** los datos (son simulados). Ejecútala y después lee el apunte de abajo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
    from scipy.stats.mstats import winsorize
except ImportError:
    print('Instala scipy con: %pip install scipy')


# semillaa
np.random.seed(142)

# 1. Crear el rango de fechas (60 meses)
fechas = pd.date_range('2021-01-01', periods=60, freq='MS')
df = pd.DataFrame({'Fecha': fechas})
df['Mes'] = df['Fecha'].dt.month
df['t'] = np.arange(len(df))

# 2. Patrón estacional en Diciembre para ganarle a la tendencia
# Marzo (3) es el punto más bajo (-40) y Diciembre (12) es por mucho el más alto (+60)
patron_estacional = {
    1: -10, 2: -25, 3: -45,  # Mínimo absoluto en Marzo
    4: -20, 5: -10, 6: 0,
    7: 5,   8: 10,  9: 15,
    10: 20, 11: 15, 12: 65  # Máximo absoluto en Diciembre (bajamos Noviembre a 15)
}
df['Estacionalidad'] = df['Mes'].map(patron_estacional)

# 3. Calcular Ventas: Tendencia + Estacionalidad + Ruido
df['Ventas'] = 100 + 1.4 * df['t'] + df['Estacionalidad'] + np.random.normal(0, 2, 60)

# 4. Valores atípicos aplicados con precisión quirúrgica
df.loc[df['Fecha'] == '2021-11-01', 'Ventas'] += 40  # Atípico Nov 2021
df.loc[df['Fecha'] == '2023-11-01', 'Ventas'] += 45  # Atípico Nov 2023

# 5. Crear etiquetas de texto para el eje X (ej: "Ene-21", "Dic-21")
# Traducimos meses clave para mantener el formato limpio
meses_es = {1:'Ene', 2:'Feb', 3:'Mar', 4:'Abr', 5:'May', 6:'Jun', 
            7:'Jul', 8:'Ago', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dic'}
df['Etiqueta_X'] = df['Fecha'].apply(lambda x: f"{meses_es[x.month]}-{str(x.year)[2:]}")





📝 **Apunte — la celda de arriba, por partes**

1. **`try: ... except ImportError:`** intenta cargar `scipy`; si no está instalado, en vez de un error feo imprime cómo instalarlo.
   - `stats` trae Box-Cox y Yeo-Johnson. `winsorize` trae la winsorización.
2. **`np.random.seed(142)`** fija la *semilla*: los números “al azar” salen **siempre iguales**. Por eso a ti te va a dar lo mismo que a la profe (por ejemplo, la primera venta = 90,259479).
3. **`fechas = pd.date_range('2021-01-01', periods=60, freq='MS')`** → 60 meses, de enero 2021 a diciembre 2025.
4. **`df['Mes'] = df['Fecha'].dt.month`** → el número del mes (1 = enero … 12 = diciembre).
5. **`df['t'] = np.arange(len(df))`** → un contador de tiempo 0, 1, 2, …, 59.
6. **`patron_estacional = {...}`** es un diccionario *mes → efecto*. `df['Mes'].map(patron_estacional)` reemplaza cada número de mes por su efecto (marzo = −45, diciembre = +65).
7. **La fórmula de las ventas**:
   $$\text{Ventas}_t = \underbrace{100}_{\text{nivel inicial}} + \underbrace{1{,}4\,t}_{\text{tendencia}} + \underbrace{\text{Estacionalidad}}_{\text{según el mes}} + \underbrace{\text{ruido}\sim N(0,\,2)}_{\text{azar}}$$
   ¡Es exactamente el **modelo aditivo** que verás en la Clase 2!
8. **`df.loc[df['Fecha'] == '2021-11-01', 'Ventas'] += 40`** → a la fila de noviembre 2021 le **suma 40**: así se “planta” un valor atípico (outlier). Lo mismo con noviembre 2023 (+45).
9. **`df['Etiqueta_X'] = ...lambda...`** crea textos tipo `"Ene-21"` para el eje X: toma el nombre del mes desde `meses_es` y los 2 últimos dígitos del año (`str(x.year)[2:]`).

🔁 **Si en el control me cambian…**
- **la semilla** (`np.random.seed(142)` → otro número): cambian todos los datos, y por lo tanto cambian los λ de Box-Cox y Yeo-Johnson (ver ⚠️ más abajo).
- **la cantidad de meses** (`periods=60`): cambia en `np.random.normal(0, 2, 60)` también — **los dos números deben coincidir**, si no sale `ValueError` (largos distintos).
- **la tendencia** (`1.4`), **el nivel** (`100`) o **el patrón estacional**: solo cambias esos números.
- **las fechas de los atípicos**: deben existir dentro del rango de fechas; si pones una fecha fuera del rango, simplemente no se modifica nada.

## Serie original
Antes de transformar conviene observar la distribución, los valores extremos y la evolución temporal.

In [ ]:
# --- Graficar usando las etiquetas de texto del eje X ---
plt.figure(figsize=(15, 6))
# Graficamos con alineación perfecta con los textos
plt.plot(df['t'], df['Ventas'], marker='o', linewidth=2, color='tab:blue', label='Ventas')

# Configurar el eje X para que muestre los meses
plt.xticks(df['t'], df['Etiqueta_X'], rotation=90, fontsize=9)

plt.title('Ventas de Supermercado: Máximo Real en Diciembre y Mínimo en Marzo')
plt.xlabel('Mes y Año')
plt.ylabel('Ventas')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

📊 **Cómo leer el gráfico de la serie original**

- **Tendencia:** la serie sube a lo largo de los 5 años (≈ +1,4 por mes ≈ +17 por año).
- **Estacionalidad:** cada año se repite la misma “ola”: **mínimo en marzo** y **máximo en diciembre** (lo dice el título).
- **Atípicos:** en **Nov-21** y **Nov-23** hay saltos que no siguen el patrón (noviembre normalmente es más bajo que diciembre y no tan alto).
- **Ruido:** pequeñas irregularidades (desviación estándar 2).

📝 **Las líneas del gráfico** (este patrón se repite en todas las clases):
`plt.figure(figsize=(15, 6))` tamaño → `plt.plot(x, y, marker='o', ...)` dibuja la línea con puntos → `plt.xticks(posiciones, textos, rotation=90)` pone las etiquetas “Ene-21” giradas → `title`, `xlabel`, `ylabel` → `legend` (cuadro con nombres) → `grid` (cuadrícula; `alpha` = transparencia) → `tight_layout` (que no se corte nada) → `show`.

## 1. Logaritmo — asimetría positiva
**Propósito:** reducir la cola derecha y comprimir los valores altos.  
**Pro:** facilita interpretar diferencias como cambios relativos/porcentuales y suele estabilizar la varianza.  
**Contra:** requiere valores estrictamente positivos y cambia la escala e interpretación.

In [ ]:
if (df['Ventas'] > 0).all():
    df['Log'] = np.log(df['Ventas'])
else:
    df['Log'] = np.nan
df[['Ventas','Log']].head()

📝 **Apunte — Logaritmo**

- `if (df['Ventas'] > 0).all():` revisa que **todos** los valores sean positivos (el logaritmo de 0 o de un negativo no existe). Si hay alguno ≤ 0, la columna queda vacía (`np.nan`).
- `np.log` es el **logaritmo natural** (base *e*).
- Ejemplo de la salida: 90,26 → 4,5027.

📊 **Cómo interpretar:** en logaritmos, una **diferencia** se lee como **cambio porcentual aproximado**:
$\ln(110) - \ln(100) = 0{,}0953 \approx 9{,}5\%$. Por eso la profe dice “facilita interpretar diferencias como cambios relativos”. Además “comprime” los valores altos (la cola derecha).

## 2. Box-Cox — varianza no constante
**Propósito:** buscar automáticamente una potencia que ayude a estabilizar la varianza y aproximar la distribución a una forma más simétrica.  
**Pro:** el parámetro $\lambda$ se estima a partir de los datos.  
**Contra:** exige datos estrictamente positivos y el resultado es menos intuitivo.

In [ ]:
if (df['Ventas'] > 0).all():
    df['BoxCox'], lambda_bc = stats.boxcox(df['Ventas'])
    print('Lambda Box-Cox:', round(lambda_bc, 4))

📝 **Apunte — Box-Cox**

Fórmula: $\;y^{(\lambda)} = \dfrac{y^{\lambda} - 1}{\lambda}\;$ (y si $\lambda = 0$, es $\ln y$).

- `stats.boxcox(...)` devuelve **dos cosas**: la serie transformada y el **λ óptimo**. Por eso a la izquierda hay dos nombres: `df['BoxCox'], lambda_bc = ...`.
- Resultado: **λ = 0,6575**.

📊 **Cómo interpretar λ** (te lo pueden preguntar):

| λ | Equivale a… |
|---|---|
| 1 | No transformar (solo corre la serie) |
| 0,5 | Raíz cuadrada |
| 0 | Logaritmo |
| −1 | Inverso |

λ = 0,6575 está **entre la raíz cuadrada (0,5) y no transformar (1)** → los datos necesitaban solo una corrección **suave**.

La celda siguiente solo **muestra la tabla completa con formato bonito** (CSS: bordes, filas grises alternadas). No calcula nada estadístico.

In [ ]:
# 2. Aplicar estilos CSS para imitar el diseño de la imagen
styled_df = df.style.set_table_styles([
    # Línea horizontal superior debajo del encabezado
    {'selector': 'th', 'props': [
        ('border-bottom', '1px solid #ddd'), 
        ('padding', '8px 12px'),
        ('font-weight', 'bold')
    ]},
    # Espaciado de las celdas
    {'selector': 'td', 'props': [
        ('padding', '8px 12px')
    ]},
    # Filas alternadas (filas impares con fondo gris muy claro)
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', '#f9f9f9')
    ]},
    # Asegurar que los índices se vean en negrita
    {'selector': 'tbody th', 'props': [
        ('font-weight', 'bold')
    ]}
])

styled_df

⚠️ **Ojo:** si te sale `AttributeError: The '.style' accessor requires jinja2`, instala con `%pip install jinja2` y reinicia el kernel. Es solo para mostrar la tabla; no afecta los cálculos.

## 3. Raíz cuadrada — dispersión moderada
**Propósito:** comprimir valores elevados de forma menos intensa que el logaritmo.  
**Pro:** sencilla y útil especialmente en variables no negativas o conteos.  
**Contra:** no admite valores negativos sin un desplazamiento y puede ser insuficiente ante asimetría fuerte.

In [ ]:
df['Raiz'] = np.sqrt(df['Ventas'].clip(lower=0))
df[['Ventas','Raiz']].head()

📝 **Apunte — Raíz cuadrada**

- `.clip(lower=0)` reemplaza cualquier valor negativo por 0 **antes** de sacar raíz (para que no salga error/NaN).
- Ejemplo: 90,26 → 9,50.
- Comprime los valores altos, pero **menos** que el logaritmo.

## 4. Yeo-Johnson — ceros o valores negativos
**Propósito:** transformar datos sin exigir positividad.  
**Pro:** acepta cero y números negativos; es más flexible que Box-Cox.  
**Contra:** la escala resultante tiene una interpretación económica menos directa.

In [ ]:
df['YeoJohnson'], lambda_yj = stats.yeojohnson(df['Ventas'])
print('Lambda Yeo-Johnson:', round(lambda_yj, 4))

In [ ]:
#2. Aplicar Yeo-Johnson con lambda = 0.6552
lambda_yj = 0.6552
df['YeoJohnson'] = stats.yeojohnson(df['Ventas'], lmbda=lambda_yj)

# 3. Aplicar estilos visuales idénticos a la imagen
styled_df = df.style.set_table_styles([
    {'selector': 'th', 'props': [
        ('border-bottom', '1px solid #ddd'), 
        ('padding', '8px 12px'),
        ('font-weight', 'bold')
    ]},
    {'selector': 'td', 'props': [
        ('padding', '8px 12px')
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', '#f9f9f9')
    ]},
    {'selector': 'tbody th', 'props': [
        ('font-weight', 'bold')
    ]}
]).format(precision=6) # Mantiene la precisión de decimales original

styled_df


📝 **Apunte — Yeo-Johnson**

- Es como Box-Cox pero **acepta ceros y negativos**.
- Primera celda: `stats.yeojohnson(df['Ventas'])` **sin** λ → devuelve serie + λ óptimo (**0,6552**, casi igual que Box-Cox porque aquí todos los datos son positivos).
- Segunda celda: `stats.yeojohnson(df['Ventas'], lmbda=lambda_yj)` **con** λ dado → devuelve **solo** la serie (por eso hay un solo nombre a la izquierda).

⚠️ **Ojo importante para el control:** en la segunda celda la profe escribió el número **a mano**: `lambda_yj = 0.6552`. Si te cambian los datos, el λ va a ser otro: tienes que **reemplazar 0.6552 por el λ nuevo** que imprimió la celda anterior (o simplemente borrar esa línea para que use el `lambda_yj` recién calculado).

## 5. Estandarización Z — escalas incomparables
**Propósito:** centrar la variable en 0 y expresarla en unidades de desviación estándar.  
**Pro:** permite comparar variables con escalas/unidades diferentes y es útil en muchos modelos.  
**Contra:** **no normaliza** la distribución y es sensible a valores atípicos.

In [ ]:
df['Z'] = (df['Ventas'] - df['Ventas'].mean()) / df['Ventas'].std(ddof=0)
print('Media Z:', round(df['Z'].mean(), 6))
print('Desv. estándar Z:', round(df['Z'].std(ddof=0), 6))

📝 **Apunte — Estandarización Z**

$$Z = \frac{x - \bar x}{s}$$

- `df['Ventas'].mean()` = promedio (aquí ≈ 144,42); `.std(ddof=0)` = desviación estándar **poblacional** (divide por *n*; con `ddof=1` dividiría por *n−1*). Aquí ≈ 39,28.
- Por construcción, la nueva columna tiene **media 0 y desviación 1** (eso es lo que imprime la celda).

📊 **Cómo interpretar un Z:** *Z = 2* significa “2 desviaciones estándar **sobre** el promedio”; *Z = −1* “1 desviación **bajo** el promedio”. Aquí el máximo es Z ≈ 2,54 (Dic-25) y el mínimo Z ≈ −2,15 (Mar-21).

⚠️ **Lo que más preguntan:** Z **no normaliza** (no cambia la forma de la distribución, es solo correr y estirar la escala) y es **sensible a atípicos** (porque el promedio y la desviación se ven afectados por ellos).

💡 **Para pensar:** el atípico de Nov-21 tiene un Z de apenas ≈ 0,60. Como la serie tiene tendencia, un valor “raro para su mes” no necesariamente es extremo respecto del promedio de **toda** la serie.

In [ ]:


# 4. Aplicar el diseño visual estilizado
styled_df = df.style.set_table_styles([
    {'selector': 'th', 'props': [
        ('border-bottom', '1px solid #ddd'), 
        ('padding', '8px 12px'),
        ('font-weight', 'bold'),
        ('text-align', 'center')
    ]},
    {'selector': 'td', 'props': [
        ('padding', '8px 12px'),
        ('text-align', 'right')
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', '#f9f9f9')
    ]},
    {'selector': 'tbody th', 'props': [
        ('font-weight', 'bold')
    ]}
]).format(precision=6)

# Mostrar la tabla final en tu Jupyter Notebook
styled_df


## 6. Winsorización — valores atípicos confirmados
**Propósito:** limitar la influencia de observaciones extremas sin eliminarlas. Aquí se limita aproximadamente el 5% de cada extremo.  
**Pro:** conserva el número de observaciones y reduce la influencia de outliers.  
**Contra:** modifica observaciones reales; el umbral debe justificarse y documentarse.

In [ ]:
df['Winsorizada'] = np.asarray(winsorize(df['Ventas'], limits=[0.05, 0.05]), dtype=float)
df[['Ventas','Winsorizada']].sort_values('Ventas', ascending=False).head(8)

📝 **Apunte — Winsorización**

- `winsorize(df['Ventas'], limits=[0.05, 0.05])` → toma el **5% más bajo** y el **5% más alto** y los **reemplaza** por el valor que está justo en el límite (no los borra).
- Con 60 datos: 5% de 60 = **3 datos por cola**.
  - Los 3 más altos (Dic-25 = 244,02; Dic-24 = 229,68; Dic-23 = 216,10) pasan a valer **206,88** (el 4.º más alto).
  - Los 3 más bajos (Mar-21, Mar-22, Feb-21) pasan a valer **85,14** (el 4.º más bajo).
- `np.asarray(..., dtype=float)` convierte el resultado (un “arreglo enmascarado”) en números normales para guardarlo como columna.
- `.sort_values('Ventas', ascending=False).head(8)` ordena de mayor a menor y muestra los 8 primeros: así se ve qué valores fueron recortados.

📊 **Cómo leer la tabla:** cuando `Ventas` ≠ `Winsorizada`, ese valor fue recortado.

💡 **Para pensar (buena respuesta de control):** aquí los recortados de arriba **no son** los atípicos de noviembre que se plantaron, sino los **diciembres de los últimos años** (que son altos por tendencia + estacionalidad). La winsorización recorta los extremos de **toda la serie**, no los “raros para su mes”. Por eso la profe dice: *el umbral debe justificarse y documentarse*.

🔁 **Si me cambian el porcentaje:** `limits=[0.10, 0.10]` = 10% por cola → 6 datos por cola. Pueden ser distintos: `limits=[0, 0.05]` recorta solo arriba.

In [ ]:


# 4. Aplicar el diseño visual estilizado
styled_df = df.style.set_table_styles([
    {'selector': 'th', 'props': [
        ('border-bottom', '1px solid #ddd'), 
        ('padding', '8px 12px'),
        ('font-weight', 'bold'),
        ('text-align', 'center')
    ]},
    {'selector': 'td', 'props': [
        ('padding', '8px 12px'),
        ('text-align', 'right')
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', '#f9f9f9')
    ]},
    {'selector': 'tbody th', 'props': [
        ('font-weight', 'bold')
    ]}
]).format(precision=6)

# Mostrar la tabla final en tu Jupyter Notebook
styled_df


## Comparación visual
Las escalas son diferentes, por lo que interesa observar principalmente cómo cambia la forma y la influencia relativa de los extremos.

In [ ]:

# ==========================================
# CELDA DE GRÁFICOS - SERIE COMPLETA COREGIDA
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import mstats

# 1. Asegurar los cálculos sobre los 60 meses de la serie completa
df['BoxCox'] = stats.boxcox(df['Ventas'], lmbda=0.6575)
df['Raiz'] = np.sqrt(df['Ventas'])
df['YeoJohnson'] = stats.yeojohnson(df['Ventas'], lmbda=0.6552)
df['Z'] = (df['Ventas'] - df['Ventas'].mean()) / df['Ventas'].std(ddof=0)
df['Winsorizada'] = np.asarray(mstats.winsorize(df['Ventas'], limits=[0.05, 0.05]), dtype=float)

# 2. Configurar los subplots para las 7 columnas (Idéntico a tu estructura original)
columnas = ['Ventas', 'Log', 'BoxCox', 'Raiz', 'YeoJohnson', 'Z', 'Winsorizada']
fig, axes = plt.subplots(len(columnas), 1, figsize=(11, 16), sharex=True)

# 3. Iterar y graficar sobre toda la línea de tiempo (t) usando tus etiquetas (Etiqueta_X)
for ax, col in zip(axes, columnas):
    # Graficamos usando 't' como eje X numérico para mantener la continuidad perfecta
    ax.plot(df['t'], df[col], color='#2b5c8f', linewidth=1.8, alpha=0.9, marker='o', markersize=3)
    ax.set_title(col, fontsize=11, fontweight='bold', loc='left', pad=4)
    ax.grid(visible=True, alpha=0.25, linestyle='--')
    ax.tick_params(axis='both', labelsize=9)

# 4. Formatear ÚNICAMENTE el eje X del último gráfico con tus etiquetas de texto para que no se superpongan
plt.xticks(df['t'], df['Etiqueta_X'], rotation=90, fontsize=8)

plt.tight_layout()
plt.show()


📝 **Apunte — la celda de gráficos**

1. Vuelve a importar todo (para que la celda funcione sola) y **recalcula** todas las transformaciones.
   ⚠️ **Ojo:** aquí los λ también están **escritos a mano** (`lmbda=0.6575` y `lmbda=0.6552`). Si cambian los datos, **cámbialos** por los nuevos (o por `lambda_bc` y `lambda_yj`).
2. `plt.subplots(len(columnas), 1, figsize=(11, 16), sharex=True)` → **7 gráficos uno debajo del otro** (7 filas, 1 columna) que comparten el eje X.
3. `for ax, col in zip(axes, columnas):` recorre cada gráfico (`ax`) junto con su columna (`col`) y dibuja una línea en cada uno.
4. Al final pone las etiquetas “Ene-21…” solo en el último gráfico.

📊 **Cómo leer los 7 gráficos** (lo que interesa es la **forma**, porque las escalas son distintas):
- **Todas tienen los picos y valles en los mismos meses**: las transformaciones son *monótonas* (no cambian el orden de los datos).
- **Log, Box-Cox, Raíz y Yeo-Johnson** “achatan” los valores altos: los picos de diciembre de los últimos años se ven **relativamente menos exagerados** que en la original. El que más comprime es el **logaritmo**.
- **Z** se ve **idéntica a la original**; solo cambian los números del eje (ahora centrada en 0). Confirma que Z no cambia la forma.
- **Winsorizada** tiene el **techo plano** (206,88) en los últimos diciembres y el **piso plano** (85,14) en los primeros marzos.

## ¿Cuál elegir?

| Problema observado | Transformación | Propósito |
|---|---|---|
| Asimetría positiva | Logaritmo | Reducir cola derecha |
| Varianza no constante | Box-Cox | Estabilizar amplitud |
| Dispersión moderada | Raíz cuadrada | Comprimir valores altos |
| Ceros o negativos | Yeo-Johnson | Transformar sin exigir positividad |
| Escalas incomparables | Estandarización Z | Centrar y escalar; no normaliza |
| Atípicos confirmados | Winsorización | Limitar su influencia con trazabilidad |

**Conclusión:** transformar no es un procedimiento automático para 'mejorar' una serie. La transformación debe responder a un problema estadístico concreto y quedar justificada y documentada.

✅ **Autoevaluación — Clase 1**

<details><summary>1. ¿Qué transformación usarías si la serie tiene ceros o números negativos?</summary>

**Yeo-Johnson**: es la única de la lista que no exige positividad. Logaritmo y Box-Cox exigen datos estrictamente positivos.
</details>

<details><summary>2. Box-Cox entregó λ = 0,05. ¿Qué significa?</summary>

Está muy cerca de 0, así que la transformación óptima es **casi un logaritmo**: los datos tenían bastante asimetría o varianza creciente.
</details>

<details><summary>3. ¿Por qué después de estandarizar la media es 0 y la desviación 1? ¿Eso hace que los datos sean normales?</summary>

Porque a cada dato se le resta el promedio (queda centrado en 0) y se divide por la desviación (queda en unidades de desviación estándar). **No** los hace normales: la forma de la distribución es la misma.
</details>

<details><summary>4. Con 100 datos y limits=[0.05, 0.05], ¿cuántos datos se modifican?</summary>

5 por cola → **10 en total** (5 abajo y 5 arriba). Se reemplazan, no se eliminan: siguen siendo 100 datos.
</details>

<details><summary>5. Si me cambian la semilla, ¿qué tengo que actualizar en el código?</summary>

Los λ escritos a mano: `lambda_yj = 0.6552` (celda de Yeo-Johnson con estilo) y `lmbda=0.6575` / `lmbda=0.6552` en la celda de gráficos. Los pongo con los nuevos λ que imprimen las celdas de Box-Cox y Yeo-Johnson.
</details>

---
# 🟩 CLASE 2 — Descomposición clásica
*Archivo de la profe: `Descomposicion_Clasica_Serie_Tiempo.ipynb`*

# Descomposición clásica de una serie de tiempo en Python

Este notebook muestra cómo descomponer una serie temporal en sus componentes principales:

- **Tendencia**
- **Estacionalidad**
- **Residuo o componente irregular**

Se trabajará con los modelos **aditivo** y **multiplicativo**.

## 1. Importar librerías

Usaremos `pandas` para manejar la serie, `matplotlib` para los gráficos y `seasonal_decompose` de `statsmodels` para realizar la descomposición.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

## 2. Crear una serie mensual de ejemplo

Se construye una serie ficticia de ventas mensuales entre enero de 2020 y diciembre de 2023.

In [ ]:
fechas = pd.date_range(
    start='2020-01-01',
    periods=48,
    freq='MS'
)

ventas = [
    120, 125, 130, 128, 135, 145, 155, 160, 150, 140, 130, 125,
    130, 135, 140, 138, 145, 158, 168, 175, 165, 152, 142, 135,
    140, 145, 150, 148, 158, 170, 182, 190, 180, 165, 155, 148,
    150, 155, 160, 158, 168, 182, 195, 205, 192, 178, 165, 158
]

serie = pd.Series(ventas, index=fechas)

serie.head()

📝 **Apunte**

- `seasonal_decompose` (de `statsmodels`) es la función que hace la descomposición.
- Aquí los datos se escriben **a mano** en una lista (`ventas = [...]`, 48 números = 4 años).
- `pd.Series(ventas, index=fechas)` une los números con las fechas. **El índice de fechas es importante**: así Python sabe que son meses.

🔁 **Si en el control me dan otros números:** reemplazo la lista `ventas`. **La cantidad de números debe ser igual a `periods`** (aquí 48). Si cambia el inicio, cambio `start='2020-01-01'`.

## 3. Visualizar la serie original

Antes de descomponer una serie siempre es recomendable observar su comportamiento gráfico.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(serie)
plt.title('Serie mensual de ventas')
plt.xlabel('Fecha')
plt.ylabel('Ventas')
plt.grid(alpha=0.3)
plt.show()

## 4. Modelo aditivo

La descomposición aditiva supone:

$$Y_t = T_t + S_t + R_t$$

donde:

- $Y_t$: serie observada
- $T_t$: tendencia
- $S_t$: estacionalidad
- $R_t$: residuo

El modelo aditivo suele ser apropiado cuando la amplitud de las fluctuaciones estacionales se mantiene aproximadamente constante a lo largo del tiempo.

In [ ]:
descomposicion_aditiva = seasonal_decompose(
    serie,
    model='additive',
    period=12
)

descomposicion_aditiva.plot()
plt.suptitle('Descomposición clásica aditiva', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### ¿Por qué `period=12`?

La serie es mensual. Si existe un patrón que se repite cada año, la periodicidad estacional es de **12 meses**.

📝 **Apunte — ¿qué hace `seasonal_decompose` por dentro?** (método clásico)

1. **Tendencia:** calcula un **promedio móvil centrado de 12 meses**. Promediar un año completo “borra” la estacionalidad y deja solo el movimiento de largo plazo.
2. **Quita la tendencia:** aditivo → $Y - T$; multiplicativo → $Y / T$.
3. **Estacionalidad:** promedia lo que quedó **por cada mes** (todos los eneros juntos, todos los febreros…). Resultado: **un mismo patrón de 12 valores que se repite igual todos los años**.
4. **Residuo:** lo que sobra: $R = Y - T - S$ (aditivo) o $R = Y / (T \cdot S)$ (multiplicativo).

📊 **Cómo leer el gráfico de 4 paneles**

| Panel | Qué es | Qué mirar |
|---|---|---|
| 1 (arriba, sin nombre o *Observed*) | La serie original | Su forma general |
| **Trend** | Tendencia | ¿Sube, baja, se estanca? Aquí **sube** todo el período |
| **Seasonal** | Estacionalidad | La “ola” que se repite. **Siempre igual cada año** en el método clásico |
| **Resid** | Residuo | Puntos alrededor de **0** (aditivo). Si se ven **al azar**, la descomposición capturó bien la estructura; si se ve un patrón, quedó algo sin explicar |

⚠️ **Ojo:** la tendencia **no existe para los primeros 6 y los últimos 6 meses** (un promedio móvil de 12 necesita 6 datos antes y 6 después). Por eso el gráfico de tendencia y residuo empieza más tarde y termina antes, y en la tabla de la siguiente celda las primeras filas salen `NaN`.

🔁 **`period`**: es cada cuánto se repite el patrón. **Mensual → 12; trimestral → 4; semanal con patrón anual → 52; diario con patrón semanal → 7.** Se necesitan **al menos 2 ciclos completos** (por ejemplo, 24 meses con `period=12`), si no, sale error.

## 5. Extraer cada componente

In [ ]:
tendencia = descomposicion_aditiva.trend
estacionalidad = descomposicion_aditiva.seasonal
residuo = descomposicion_aditiva.resid

tabla = pd.DataFrame({
    'Serie': serie,
    'Tendencia': tendencia,
    'Estacionalidad': estacionalidad,
    'Residuo': residuo
})

tabla.round(2).head(20)

📝 **Apunte**

- `.trend`, `.seasonal` y `.resid` sacan cada componente como una serie.
- `pd.DataFrame({...})` los junta en una tabla con la serie original.
- `tabla.round(2).head(20)` muestra las primeras 20 filas redondeadas.

📊 **Cómo leer la tabla (comprobación):** en cualquier fila sin `NaN`, **Tendencia + Estacionalidad + Residuo = Serie**.
Ejemplo julio 2020: 137,33 + 19,44 + (−1,77) = 155 ✔️

## 6. Interpretación de los componentes

- **Serie observada:** valor total registrado en cada período.
- **Tendencia:** movimiento de largo plazo de la serie.
- **Estacionalidad:** patrón sistemático que se repite cada cierto número de períodos.
- **Residuo:** variación que no es explicada por la tendencia ni por la estacionalidad.

Por ejemplo, si la componente estacional de un mes es `+15`, ese mes presenta aproximadamente 15 unidades adicionales respecto del nivel explicado por la tendencia.

## 7. Modelo multiplicativo

El modelo multiplicativo supone:

$$Y_t = T_t \times S_t \times R_t$$

Suele ser más apropiado cuando la amplitud de la estacionalidad aumenta o disminuye junto con el nivel de la serie.

In [ ]:
descomposicion_multiplicativa = seasonal_decompose(
    serie,
    model='multiplicative',
    period=12
)

descomposicion_multiplicativa.plot()
plt.suptitle('Descomposición clásica multiplicativa', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

📊 **Cómo leer el gráfico multiplicativo**

Se ve casi igual que el aditivo, **pero cambian las escalas** de los paneles *Seasonal* y *Resid*:
- **Seasonal** ahora son **factores** alrededor de **1** (por ejemplo 1,167 = 16,7% sobre la tendencia; 0,886 = 11,4% bajo la tendencia).
- **Resid** ahora está alrededor de **1** (no de 0).

## 8. ¿Aditivo o multiplicativo?

### Modelo aditivo

Usar cuando las oscilaciones estacionales tienen una magnitud relativamente constante.

$$Y_t = T_t + S_t + R_t$$

### Modelo multiplicativo

Usar cuando las oscilaciones estacionales crecen o disminuyen proporcionalmente con el nivel de la serie.

$$Y_t = T_t \times S_t \times R_t$$

## 9. Comparación simple de la estacionalidad

In [ ]:
estacionalidad_promedio = pd.DataFrame({
    'Mes': ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'],
    'Efecto_estacional_aditivo': descomposicion_aditiva.seasonal.iloc[:12].values,
    'Factor_estacional_multiplicativo': descomposicion_multiplicativa.seasonal.iloc[:12].values
})

estacionalidad_promedio.round(3)

📊 **Cómo leer la tabla de estacionalidad** (con los datos de la profe)

| Mes | Aditivo | Multiplicativo | Lectura |
|---|---|---|---|
| Ago (máximo) | +25,27 | 1,167 | En agosto las ventas están ~25 unidades (≈ 16,7%) **sobre** la tendencia |
| Dic (mínimo) | −17,24 | 0,886 | En diciembre están ~17 unidades (≈ 11,4%) **bajo** la tendencia |

- Los 12 efectos **aditivos suman ≈ 0**; los 12 factores **multiplicativos promedian ≈ 1**. (Lo que un mes sube, otros lo bajan.)
- Para pasar de factor a %: $(\text{factor} - 1)\times 100$.

⚠️ **Ojo:** `.iloc[:12]` toma los **primeros 12 meses** de la serie y les pone las etiquetas `'Ene'…'Dic'`. Eso solo es correcto **si la serie empieza en enero**. Si te dan una serie que empieza, por ejemplo, en abril, las etiquetas quedarían corridas.

📝 **¿Aditivo o multiplicativo para ESTA serie?** Mira el rango de cada año (máximo − mínimo):

| Año | Rango (unidades) | Promedio anual | Rango / promedio |
|---|---|---|---|
| 2020 | 40 | 136,9 | 29% |
| 2021 | 45 | 148,6 | 30% |
| 2022 | 50 | 160,9 | 31% |
| 2023 | 55 | 172,2 | 32% |

La amplitud **crece en unidades a medida que sube el nivel**, pero como **porcentaje** se mantiene casi igual → la estacionalidad es **proporcional al nivel** → el **multiplicativo** es más apropiado. Ese es exactamente el criterio de la sección 8 de la profe.

## 10. Idea clave para recordar

La descomposición clásica permite escribir una serie temporal como la combinación de componentes que tienen interpretaciones distintas. Esto facilita comprender si los cambios observados corresponden a una tendencia de largo plazo, a un patrón estacional o a movimientos irregulares.

En análisis reales, después de esta descomposición se puede avanzar hacia métodos como **STL**, suavizamiento exponencial, ARIMA o SARIMA.

✅ **Autoevaluación — Clase 2**

<details><summary>1. Escribe el modelo aditivo y el multiplicativo y di cuándo se usa cada uno.</summary>

Aditivo: $Y_t = T_t + S_t + R_t$ → cuando las oscilaciones estacionales tienen **tamaño constante** en unidades.
Multiplicativo: $Y_t = T_t \times S_t \times R_t$ → cuando las oscilaciones **crecen o disminuyen proporcionalmente** al nivel.
</details>

<details><summary>2. El efecto estacional aditivo de octubre es +10,7. ¿Cómo se interpreta?</summary>

En octubre la serie está, en promedio, **unas 10,7 unidades sobre** lo que indica la tendencia.
</details>

<details><summary>3. El factor multiplicativo de abril es 0,985. ¿Cómo se interpreta?</summary>

En abril la serie está ≈ **1,5% bajo** la tendencia ((0,985 − 1) × 100 = −1,5%).
</details>

<details><summary>4. ¿Por qué aparecen NaN en la tendencia al principio y al final?</summary>

Porque la tendencia es un promedio móvil centrado de 12 meses: necesita 6 datos antes y 6 después. En los extremos no los hay.
</details>

<details><summary>5. Me dan datos trimestrales. ¿Qué cambio?</summary>

`period=4` en `seasonal_decompose` y `freq='QS'` en `pd.date_range` (trimestre). Además la tabla de 12 meses ya no aplica (serían 4 trimestres: `.iloc[:4]`).
</details>

---
# 🟧 CLASE 3 — Suavizamiento exponencial, Holt y Holt-Winters
*Archivo de la profe: `01_suavizamiento_exponencial_holt_winters_banco_mundial_2.ipynb`*

⚠️ Esta clase **descarga datos de internet** (API del Banco Mundial). Necesitas conexión.

# Suavizamiento exponencial, Holt y Holt-Winters
## Profesor: Beatriz Berríos P.

## Teoría, aplicación y práctica con datos del Banco Mundial

Este notebook descarga mediante la API V2 del Banco Mundial el **tipo de cambio nominal mensual de Chile**, indicador `DPANUSLCU`, perteneciente a Global Economic Monitor.

Se comparan cuatro especificaciones:

1. **Suavizamiento exponencial simple, SES:** modela únicamente el nivel.
2. **Holt:** incorpora nivel y tendencia.
3. **Holt-Winters aditivo:** incorpora una estacionalidad de magnitud aproximadamente constante.
4. **Holt-Winters multiplicativo:** incorpora una estacionalidad proporcional al nivel.

> En muchos textos, “método de Winters” y “Holt-Winters” identifican la misma familia estacional. La distinción importante es si el componente estacional se representa de forma aditiva o multiplicativa.

### Paquetes requeridos

Si algún paquete no está instalado, ejecute en una celda:

```python
%pip install pandas numpy matplotlib requests statsmodels
```

La API del Banco Mundial no requiere una clave.

## Formulación de los modelos

### Suavizamiento exponencial simple

$$\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1}$$

$$\widehat y_{t+h\mid t}=\ell_t$$

Es adecuado cuando la serie fluctúa alrededor de un nivel relativamente estable, sin tendencia ni estacionalidad persistentes.

### Método de Holt

$$\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})$$

$$b_t=\beta(\ell_t-\ell_{t-1})+(1-\beta)b_{t-1}$$

$$\widehat y_{t+h\mid t}=\ell_t+h b_t$$

Holt incorpora un nivel local y una pendiente. La tendencia amortiguada reduce gradualmente su efecto en horizontes largos.

### Holt-Winters aditivo

$$\widehat y_{t+h\mid t}=\ell_t+h b_t+s_{t-m+h_m}$$

Se utiliza cuando la amplitud estacional se mantiene aproximadamente constante en unidades.

### Holt-Winters multiplicativo

$$\widehat y_{t+h\mid t}=(\ell_t+h b_t)s_{t-m+h_m}$$

Se utiliza cuando la amplitud estacional crece o disminuye proporcionalmente con el nivel. Requiere valores positivos.

### Interpretación de los parámetros

- $\alpha$ controla la actualización del nivel.
- $\beta$ controla la actualización de la tendencia.
- $\gamma$ controla la actualización del componente estacional.
- Valores cercanos a 1 producen mayor reacción ante observaciones nuevas.
- Valores cercanos a 0 producen mayor suavidad y memoria histórica.

📝 **Apunte — las fórmulas en palabras**

**SES:** $\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1}$
“El nivel nuevo es una **mezcla** entre el dato de hoy ($y_t$) y el nivel anterior ($\ell_{t-1}$)”. $\alpha$ decide cuánto pesa el dato nuevo.
Ejemplo con $\alpha = 0{,}5$: nivel anterior 100, dato nuevo 110 → nivel nuevo = 0,5·110 + 0,5·100 = **105**.
El pronóstico es **el último nivel para todos los meses futuros** → **una línea plana**.

**Holt:** agrega una **pendiente** $b_t$ (cuánto sube o baja por período). Pronóstico = nivel + h · pendiente → **una línea inclinada**. Con **tendencia amortiguada** (`damped_trend=True`) la pendiente se va “apagando” y la línea se va aplanando con el horizonte.

**Holt-Winters:** agrega la **estacionalidad** $s$ → el pronóstico **ondula** según el mes.
- **Aditivo:** se **suma** el efecto del mes (ej.: +30 cada diciembre).
- **Multiplicativo:** se **multiplica** por el factor del mes (ej.: ×1,15 = +15% cada diciembre).

| Parámetro | Suaviza… | Cerca de 1 | Cerca de 0 |
|---|---|---|---|
| $\alpha$ (alpha) | nivel | reacciona mucho a lo último | muy suave, mucha memoria |
| $\beta$ (beta) | tendencia | la pendiente cambia rápido | pendiente casi fija |
| $\gamma$ (gamma) | estacionalidad | el patrón estacional se actualiza rápido | patrón estacional casi fijo |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing,
    Holt,
    ExponentialSmoothing,
)


📝 **Apunte:** `requests` sirve para pedir datos a internet. De `statsmodels` se importan las **3 funciones de modelos**: `SimpleExpSmoothing` (SES), `Holt` y `ExponentialSmoothing` (Holt-Winters).

## 1. Descarga de la serie

La consulta solicita observaciones mensuales de Chile desde Global Economic Monitor. Se comprueban valores faltantes y solo se permite interpolar brechas cortas de hasta dos meses. Una brecha extensa debe investigarse antes de modelar.

In [ ]:
url_gem = "https://api.worldbank.org/v2/country/CHL/indicator/DPANUSLCU"

parametros = {
    "format": "json",
    "source": 15,
    "frequency": "M",
    "date": "1995M01:2026M12",
    "per_page": 5000,
}

respuesta = requests.get(url_gem, params=parametros, timeout=60)
respuesta.raise_for_status()
contenido = respuesta.json()

if not isinstance(contenido, list) or len(contenido) < 2 or contenido[1] is None:
    raise RuntimeError(
        "La API no devolvió observaciones. Revise la conexión, el código "
        "DPANUSLCU y la disponibilidad de la fuente GEM 15."
    )

datos = pd.DataFrame(contenido[1])
datos = datos.loc[:, ["date", "value"]].copy()
datos = datos.dropna(subset=["value"])

# La API representa los meses como 2024M01. Se cambia M por guion.
datos["fecha"] = pd.to_datetime(
    datos["date"].astype(str).str.replace("M", "-", regex=False),
    format="%Y-%m",
)

datos["tipo_cambio"] = pd.to_numeric(datos["value"], errors="coerce")
datos = datos.dropna(subset=["tipo_cambio"])
datos = datos.sort_values("fecha")

serie_original = datos.set_index("fecha")["tipo_cambio"]
serie_original = serie_original[~serie_original.index.duplicated(keep="last")]
serie_original = serie_original.asfreq("MS")

faltantes_originales = int(serie_original.isna().sum())

# Solo se interpolan brechas cortas de hasta dos meses.
serie = serie_original.interpolate(method="time", limit=2)

if serie.isna().any():
    raise RuntimeError(
        "Persisten brechas mayores a dos meses. Revise los datos antes de "
        "ajustar modelos estacionales; no conviene interpolarlas automáticamente."
    )

print("Fuente: Banco Mundial, Global Economic Monitor (GEM)")
print("Indicador: DPANUSLCU")
print("Primera fecha:", serie.index.min().strftime("%Y-%m"))
print("Última fecha:", serie.index.max().strftime("%Y-%m"))
print("Observaciones:", len(serie))
print("Valores faltantes originales:", faltantes_originales)
print(serie.tail())


📝 **Apunte — la descarga, paso a paso**

1. **`url_gem`**: la dirección de la API. **`CHL`** = Chile (código ISO de 3 letras). **`DPANUSLCU`** = el indicador (tipo de cambio, moneda local por US$).
2. **`parametros`**: `format` json, `source: 15` (base Global Economic Monitor), `frequency: "M"` (mensual), `date` (rango de fechas) y `per_page` (cuántos datos traer como máximo).
3. **`requests.get(...)`** pide los datos; **`raise_for_status()`** lanza error si la página falló; **`.json()`** convierte la respuesta en listas/diccionarios de Python.
4. **El `if not isinstance...`** revisa que la respuesta tenga datos. La API devuelve una lista de 2 partes: `[información, datos]`; por eso se usa `contenido[1]`.
5. **Limpieza:**
   - se quedan solo las columnas `date` y `value`; se botan filas vacías (`dropna`);
   - la fecha viene como `"2024M01"`: se cambia la `M` por `-` → `"2024-01"` y se convierte a fecha;
   - `pd.to_numeric(..., errors="coerce")`: lo que no sea número pasa a `NaN`;
   - `sort_values("fecha")`: ordena de la más antigua a la más nueva (la API las trae al revés).
6. **`set_index("fecha")["tipo_cambio"]`** → una serie con fechas como índice.
7. **`~...duplicated(keep="last")`** elimina fechas repetidas.
8. **`asfreq("MS")`** obliga a que existan **todos los meses**; si falta uno, aparece como `NaN`.
9. **`interpolate(method="time", limit=2)`** rellena huecos de **máximo 2 meses** seguidos (uniendo con una línea los vecinos). Si quedan huecos más largos → **error a propósito**: la profe dice que no conviene inventar tantos datos.
10. Los `print` muestran fuente, primera/última fecha, cantidad de datos (380 con los datos de la profe: 1995-01 a 2026-08) y los últimos 5 valores.

🔁 **Si en el control me cambian…**
- **el país** → cambio `CHL` en `url_gem` (por ejemplo `PER` Perú, `COL` Colombia, `MEX` México, `BRA` Brasil). ¡Eso es el **ejercicio 6** de la práctica! Si la API no devuelve datos para ese país, sale el `RuntimeError` de la profe → prueba otro.
- **el período** → cambio `"date": "1995M01:2026M12"`.
- **el indicador** → cambio `DPANUSLCU` en la URL (y los textos de los gráficos).
- Recuerda cambiar los **títulos** que dicen “Chile” y la etiqueta del eje Y si cambian las unidades.

## 2. Exploración y separación temporal

Se grafica la serie completa. Los últimos 24 meses se reservan como muestra de prueba. Así, las métricas evalúan pronósticos realizados sobre observaciones que el modelo no utilizó durante su estimación.

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(serie.index, serie.values, color="#123047", linewidth=1.8)
plt.title("Chile: tipo de cambio nominal mensual")
plt.xlabel("Fecha")
plt.ylabel("Moneda local por US$")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# El período estacional es 12 porque existen 12 meses por año.
m = 12

# Se reservan los últimos 24 meses para una evaluación fuera de muestra.
h_prueba = 24
entrenamiento = serie.iloc[:-h_prueba]
prueba = serie.iloc[-h_prueba:]

print("\nTamaño de entrenamiento:", len(entrenamiento))
print("Tamaño de prueba:", len(prueba))


📊 **Cómo leer el gráfico del tipo de cambio**

- Tendencia de largo plazo **al alza** (de ~400 pesos por dólar en 1995 a ~900 en 2026), con períodos de subida y de bajada.
- **No se ve una estacionalidad clara** a simple vista: el dólar no sube “todos los diciembres”. Esto será importante para entender los resultados (spoiler: la estacionalidad aporta poco).

📝 **Apunte — la separación entrenamiento / prueba**

- **`m = 12`**: período estacional (12 meses por año).
- **`h_prueba = 24`**: los **últimos 24 meses se esconden** al modelo.
- **`entrenamiento = serie.iloc[:-h_prueba]`** → todo **menos** los últimos 24 (356 datos): con esto se estima el modelo.
- **`prueba = serie.iloc[-h_prueba:]`** → **solo** los últimos 24: con esto se evalúa.

Es como estudiar con unas guías y dar la prueba con preguntas nuevas: así se mide si el modelo **pronostica**, no si “memorizó”. Eso es **evaluación fuera de muestra**.

## 3. Suavizamiento exponencial simple

SES establece una línea base. Si Holt o Holt-Winters no mejoran su desempeño, la tendencia o estacionalidad añadidas podrían no aportar suficiente información fuera de muestra.

In [ ]:
# Adecuado para una serie que fluctúa alrededor de un nivel relativamente
# estable. No incorpora tendencia ni estacionalidad.
modelo_ses = SimpleExpSmoothing(
    entrenamiento,
    initialization_method="estimated",
)

ajuste_ses = modelo_ses.fit(optimized=True)
pronostico_ses = ajuste_ses.forecast(h_prueba)


📝 **Apunte — el patrón que se repite en TODOS los modelos**

```
modelo = Funcion(entrenamiento, opciones...)   # 1. Definir el modelo
ajuste = modelo.fit(optimized=True)            # 2. Estimarlo (busca los mejores alpha, beta, gamma)
pronostico = ajuste.forecast(h_prueba)         # 3. Pronosticar h_prueba meses hacia adelante
```

- `initialization_method="estimated"`: los valores iniciales (nivel inicial, etc.) también se estiman.
- `fit(optimized=True)`: elige los parámetros que **minimizan el error dentro del entrenamiento**.
- SES no tiene tendencia ni estacionalidad: es la **línea base**. Si un modelo más complejo no le gana, la complejidad no está sirviendo.

## 4. Método de Holt

Holt permite que el nivel cambie mediante una tendencia. Se emplea una tendencia amortiguada para evitar que la pendiente se prolongue indefinidamente con la misma intensidad.

In [ ]:
# Incorpora una tendencia. Se utiliza tendencia amortiguada para evitar que
# una pendiente estimada se prolongue indefinidamente con igual intensidad.
modelo_holt = Holt(
    entrenamiento,
    damped_trend=True,
    initialization_method="estimated",
)

ajuste_holt = modelo_holt.fit(optimized=True)
pronostico_holt = ajuste_holt.forecast(h_prueba)


## 5. Holt-Winters aditivo

El componente estacional añade una cantidad. Por ejemplo, un aumento de 30 unidades cada diciembre corresponde a una interpretación aditiva.

In [ ]:
# La estacionalidad aditiva representa aumentos o disminuciones de tamaño
# aproximadamente constante: por ejemplo, +30 unidades cada diciembre.
modelo_winters_aditivo = ExponentialSmoothing(
    entrenamiento,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=m,
    initialization_method="estimated",
)

ajuste_winters_aditivo = modelo_winters_aditivo.fit(optimized=True)
pronostico_winters_aditivo = ajuste_winters_aditivo.forecast(h_prueba)


## 6. Holt-Winters multiplicativo

El componente estacional actúa como un factor. Por ejemplo, un aumento recurrente de 15% corresponde a una interpretación multiplicativa.

In [ ]:
# La estacionalidad multiplicativa representa efectos porcentuales. Su uso
# exige observaciones positivas, condición que cumple el tipo de cambio.
modelo_holt_winters_multiplicativo = ExponentialSmoothing(
    entrenamiento,
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=m,
    initialization_method="estimated",
)

ajuste_holt_winters_multiplicativo = (
    modelo_holt_winters_multiplicativo.fit(optimized=True)
)
pronostico_holt_winters_multiplicativo = (
    ajuste_holt_winters_multiplicativo.forecast(h_prueba)
)


📝 **Apunte — las opciones de `ExponentialSmoothing`**

| Opción | Valores | Significado |
|---|---|---|
| `trend` | `"add"` / `None` | con tendencia aditiva / sin tendencia |
| `damped_trend` | `True` / `False` | tendencia amortiguada (se aplana) / no |
| `seasonal` | `"add"` / `"mul"` / `None` | estacionalidad aditiva / multiplicativa / sin estacionalidad |
| `seasonal_periods` | `m` = 12 | largo del ciclo estacional |

⚠️ **Ojo:** `seasonal="mul"` necesita datos **positivos** (el tipo de cambio lo es). Con ceros o negativos da error.

## 7. Comparación gráfica fuera de muestra

La línea vertical marca el comienzo del período de prueba. Un buen modelo debe seguir el nivel y los cambios relevantes sin reaccionar excesivamente al ruido.

In [ ]:
comparacion = pd.DataFrame(index=prueba.index)
comparacion["Real"] = prueba
comparacion["SES"] = np.asarray(pronostico_ses)
comparacion["Holt"] = np.asarray(pronostico_holt)
comparacion["Winters aditivo"] = np.asarray(pronostico_winters_aditivo)
comparacion["Holt-Winters multiplicativo"] = np.asarray(
    pronostico_holt_winters_multiplicativo
)

plt.figure(figsize=(14, 6))
plt.plot(
    entrenamiento.index[-48:],
    entrenamiento.iloc[-48:],
    color="#7A8790",
    label="Entrenamiento",
)
plt.plot(prueba.index, prueba, color="#123047", linewidth=2.8, label="Real")
plt.plot(comparacion.index, comparacion["SES"], label="SES")
plt.plot(comparacion.index, comparacion["Holt"], label="Holt")
plt.plot(
    comparacion.index,
    comparacion["Winters aditivo"],
    label="Winters aditivo",
)
plt.plot(
    comparacion.index,
    comparacion["Holt-Winters multiplicativo"],
    label="Holt-Winters multiplicativo",
)
plt.axvline(prueba.index[0], color="#C84A4A", linestyle="--", alpha=0.8)
plt.title("Comparación de pronósticos en los últimos 24 meses")
plt.xlabel("Fecha")
plt.ylabel("Moneda local por US$")
plt.legend(ncol=2)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


📝 **Apunte**

- `comparacion` es una tabla con índice = fechas de prueba y una columna por modelo. `np.asarray(...)` pega los pronósticos **por posición** (fila 1 con fila 1…).
- El gráfico: **gris** = últimos 48 meses de entrenamiento (contexto), **azul oscuro grueso** = lo real, **4 líneas de colores** = pronósticos, **línea roja punteada vertical** = donde empieza la prueba.

📊 **Cómo leer el gráfico de comparación**

- **SES** = línea **plana** (repite el último nivel).
- **Holt** = línea **inclinada** que se va aplanando (amortiguada).
- **Winters aditivo / multiplicativo** = líneas que **ondulan** levemente (estacionalidad).
- El mejor es el que se mantiene **más cerca de la línea azul oscura** durante todo el período. Como el tipo de cambio se mueve casi al azar, **ningún modelo anticipa los saltos**: todos quedan cerca del último valor conocido.

## 8. Métricas de evaluación

- **MAE:** error absoluto medio en las unidades originales.
- **RMSE:** penaliza con mayor intensidad los errores grandes.
- **MAPE:** expresa el error medio en porcentaje; no resulta apropiado si existen ceros o valores cercanos a cero.

La selección principal se realiza con RMSE, pero también se deben revisar residuos y estabilidad.

In [ ]:
# MAE: error absoluto medio, en las unidades originales.
# RMSE: penaliza con mayor fuerza los errores grandes.
# MAPE: error porcentual medio. No debe usarse si la serie contiene ceros.

error_ses = prueba.to_numpy() - comparacion["SES"].to_numpy()
error_holt = prueba.to_numpy() - comparacion["Holt"].to_numpy()
error_wa = prueba.to_numpy() - comparacion["Winters aditivo"].to_numpy()
error_hwm = (
    prueba.to_numpy()
    - comparacion["Holt-Winters multiplicativo"].to_numpy()
)

resultados = pd.DataFrame(
    {
        "MAE": [
            np.mean(np.abs(error_ses)),
            np.mean(np.abs(error_holt)),
            np.mean(np.abs(error_wa)),
            np.mean(np.abs(error_hwm)),
        ],
        "RMSE": [
            np.sqrt(np.mean(error_ses**2)),
            np.sqrt(np.mean(error_holt**2)),
            np.sqrt(np.mean(error_wa**2)),
            np.sqrt(np.mean(error_hwm**2)),
        ],
        "MAPE (%)": [
            100 * np.mean(np.abs(error_ses / prueba.to_numpy())),
            100 * np.mean(np.abs(error_holt / prueba.to_numpy())),
            100 * np.mean(np.abs(error_wa / prueba.to_numpy())),
            100 * np.mean(np.abs(error_hwm / prueba.to_numpy())),
        ],
    },
    index=[
        "SES",
        "Holt",
        "Winters aditivo",
        "Holt-Winters multiplicativo",
    ],
)

resultados = resultados.sort_values("RMSE")

print("\nMétricas fuera de muestra")
print(resultados.round(3))
print("\nMejor modelo según RMSE:", resultados.index[0])


📝 **Apunte — las métricas**

Error de cada mes: $e = \text{real} - \text{pronóstico}$ (eso es `prueba.to_numpy() - comparacion[...]`).

| Métrica | Código | Fórmula | Se lee como |
|---|---|---|---|
| MAE | `np.mean(np.abs(e))` | promedio de \|e\| | “en promedio me equivoco X pesos” |
| RMSE | `np.sqrt(np.mean(e**2))` | raíz del promedio de e² | igual que MAE pero **castiga más los errores grandes** |
| MAPE | `100 * np.mean(np.abs(e / real))` | promedio de \|e/real\| en % | “en promedio me equivoco X %” |

**Ejemplo a mano** (errores 10, −20, 5): MAE = (10+20+5)/3 = **11,67**; RMSE = √((100+400+25)/3) = √175 = **13,23**. Siempre RMSE ≥ MAE; si son muy distintos, hubo algún error grande.

- `resultados.sort_values("RMSE")` ordena **de menor a mayor** RMSE → el **mejor queda primero**.
- `resultados.index[0]` es el **nombre** del primero = el ganador.

📊 **Cómo interpretar los resultados de la profe (h_prueba = 24)**

| Modelo | MAE | RMSE | MAPE |
|---|---|---|---|
| **Winters aditivo** | 25,53 | **32,04** | 2,72% |
| SES | 26,36 | 32,56 | 2,82% |
| Holt | 26,87 | 32,89 | 2,86% |
| HW multiplicativo | 29,12 | 35,15 | 3,12% |

- Gana **Winters aditivo**: se equivoca en promedio ~25,5 pesos por dólar (≈ 2,7%).
- **Pero la diferencia con SES es mínima** (32,04 vs 32,56 = medio peso). Agregar tendencia y estacionalidad casi no mejora: el tipo de cambio se comporta casi como un **paseo aleatorio** (lo mejor que se puede hacer es parecido a “el próximo mes será como el último”).

## 9. Parámetros estimados

Los valores estimados de $\alpha$, $\beta$ y $\gamma$ muestran cuánto peso asigna el modelo a la información reciente. Un parámetro pequeño no significa que el modelo sea incorrecto: puede indicar que el componente evoluciona lentamente.

In [ ]:
parametros_estimados = pd.DataFrame(
    {
        "SES": [
            ajuste_ses.params.get("smoothing_level", np.nan),
            np.nan,
            np.nan,
        ],
        "Holt": [
            ajuste_holt.params.get("smoothing_level", np.nan),
            ajuste_holt.params.get("smoothing_trend", np.nan),
            np.nan,
        ],
        "Winters aditivo": [
            ajuste_winters_aditivo.params.get("smoothing_level", np.nan),
            ajuste_winters_aditivo.params.get("smoothing_trend", np.nan),
            ajuste_winters_aditivo.params.get("smoothing_seasonal", np.nan),
        ],
        "Holt-Winters multiplicativo": [
            ajuste_holt_winters_multiplicativo.params.get(
                "smoothing_level", np.nan
            ),
            ajuste_holt_winters_multiplicativo.params.get(
                "smoothing_trend", np.nan
            ),
            ajuste_holt_winters_multiplicativo.params.get(
                "smoothing_seasonal", np.nan
            ),
        ],
    },
    index=["alpha: nivel", "beta: tendencia", "gamma: estacionalidad"],
)

print("\nParámetros de suavizamiento")
print(parametros_estimados.round(4))


📝 **Apunte**

- `ajuste.params` es un diccionario con los parámetros estimados. `.get("smoothing_level", np.nan)` saca alpha; si el modelo no tiene ese parámetro, pone `NaN` (por eso SES tiene NaN en beta y gamma).
- Nombres: `smoothing_level` = α, `smoothing_trend` = β, `smoothing_seasonal` = γ.

📊 **Cómo interpretar los parámetros de la profe**

- **α ≈ 1** en todos (0,995; 1,000; 0,986; 0,960) → el nivel prácticamente **es el último dato**: el modelo casi “olvida” el pasado. Típico de tipo de cambio (paseo aleatorio).
- **β pequeño** (0,19 en Holt; 0,03 y 0,007 en Winters) → la pendiente cambia lento.
- **γ ≈ 0** (0,005 y 0,04) → el patrón estacional casi **no se actualiza**: la estacionalidad es débil/estable.
- Como dice la profe: un parámetro pequeño **no** significa que el modelo esté mal; significa que ese componente evoluciona lentamente.

## 10. Pronóstico final

Después de comparar los modelos en la muestra de prueba, cada especificación se vuelve a estimar usando toda la información disponible. El gráfico presenta el pronóstico del modelo con menor RMSE.

In [ ]:
h_futuro = 12

ses_final = SimpleExpSmoothing(
    serie,
    initialization_method="estimated",
).fit(optimized=True)

holt_final = Holt(
    serie,
    damped_trend=True,
    initialization_method="estimated",
).fit(optimized=True)

wa_final = ExponentialSmoothing(
    serie,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=m,
    initialization_method="estimated",
).fit(optimized=True)

hwm_final = ExponentialSmoothing(
    serie,
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=m,
    initialization_method="estimated",
).fit(optimized=True)

fecha_futura = pd.date_range(
    serie.index.max() + pd.offsets.MonthBegin(1),
    periods=h_futuro,
    freq="MS",
)

pronosticos_futuros = pd.DataFrame(index=fecha_futura)
pronosticos_futuros["SES"] = np.asarray(ses_final.forecast(h_futuro))
pronosticos_futuros["Holt"] = np.asarray(holt_final.forecast(h_futuro))
pronosticos_futuros["Winters aditivo"] = np.asarray(
    wa_final.forecast(h_futuro)
)
pronosticos_futuros["Holt-Winters multiplicativo"] = np.asarray(
    hwm_final.forecast(h_futuro)
)

mejor_modelo = resultados.index[0]

plt.figure(figsize=(14, 6))
plt.plot(serie.index[-60:], serie.iloc[-60:], color="#123047", label="Observado")
plt.plot(
    pronosticos_futuros.index,
    pronosticos_futuros[mejor_modelo],
    color="#E67E22",
    linewidth=2.5,
    label=f"Pronóstico: {mejor_modelo}",
)
plt.axvline(serie.index.max(), color="#C84A4A", linestyle="--")
plt.title("Pronóstico de 12 meses con el modelo seleccionado")
plt.xlabel("Fecha")
plt.ylabel("Moneda local por US$")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("\nPronósticos futuros")
print(pronosticos_futuros[[mejor_modelo]].round(2))


📝 **Apunte — pronóstico final**

1. Después de **elegir** el modelo con la prueba, se vuelven a estimar los 4 modelos con **toda la serie** (`serie`, no `entrenamiento`), para usar también los datos más recientes.
2. `h_futuro = 12` → pronostica 12 meses.
3. `fecha_futura`: 12 fechas empezando el mes siguiente al último dato (`pd.offsets.MonthBegin(1)` = “el próximo inicio de mes”).
4. `mejor_modelo = resultados.index[0]` → el nombre del ganador; se usa para elegir **qué columna graficar**.

📊 **Cómo leer el gráfico final:** azul oscuro = últimos 60 meses observados; **naranja** = pronóstico; línea roja = donde terminan los datos. Con Winters aditivo el pronóstico **ondula suavemente** entre ~905 y ~925. Si ganara SES, sería una **línea recta horizontal** (todos los meses el mismo valor).

⚠️ **Ojo:** `mejor_modelo` sale de la **última** tabla `resultados` que ejecutaste. Si cambias `h_prueba` y no vuelves a correr la celda de métricas, el “mejor” puede estar desactualizado.

🔁 **Si me piden pronosticar 6 o 24 meses:** cambio `h_futuro`.

---
# 🟥 ACTIVIDAD PRÁCTICA (posible modelo del control)

La profe resolvió la práctica **copiando sus propias celdas y cambiando un número**. Esa es la estrategia para el control: **identifica qué celda sirve, cópiala y cambia solo lo necesario**.

## 11. Actividad práctica

La práctica permite comprobar que la selección puede cambiar cuando se modifica el horizonte, la tendencia o la estructura estacional. Las respuestas deben incluir evidencia gráfica, métricas y una interpretación económica.

In [ ]:
print(
    """
PRÁCTICA
========
1. Cambie h_prueba de 24 a 12 meses. ¿Cambia el modelo ganador?
2. Ajuste Holt sin amortiguación: damped_trend=False. Compare el RMSE.
3. Compare la amplitud estacional en unidades y como porcentaje del nivel.
4. Justifique si corresponde estacionalidad aditiva o multiplicativa.
5. Grafique los residuos del modelo ganador y calcule su FAC.
6. Repita el análisis para otro país disponible en GEM.

Preguntas de interpretación
---------------------------
- ¿Qué información ignora SES que sí incorpora Holt?
- ¿Qué representa gamma en Holt-Winters?
- ¿Por qué el modelo multiplicativo exige valores positivos?
- ¿Por qué la evaluación debe hacerse fuera de muestra?
- ¿El menor RMSE garantiza residuos sin autocorrelación? Explique.
"""
)


### 1. Cambie h_prueba de 24 a 12 meses. ¿Cambia el modelo ganador?

In [ ]:
# Se reservan los últimos 12 meses para una evaluación fuera de muestra.
h_prueba = 12
entrenamiento = serie.iloc[:-h_prueba]
prueba = serie.iloc[-h_prueba:]

print("\nTamaño de entrenamiento:", len(entrenamiento))
print("Tamaño de prueba:", len(prueba))

In [ ]:
# Métodos

#1. SES
# Adecuado para una serie que fluctúa alrededor de un nivel relativamente
# estable. No incorpora tendencia ni estacionalidad.
modelo_ses = SimpleExpSmoothing(
    entrenamiento,
    initialization_method="estimated",
)

ajuste_ses = modelo_ses.fit(optimized=True)
pronostico_ses = ajuste_ses.forecast(h_prueba)

#2. Holt
# Incorpora una tendencia. Se utiliza tendencia amortiguada para evitar que
# una pendiente estimada se prolongue indefinidamente con igual intensidad.
modelo_holt = Holt(
    entrenamiento,
    damped_trend=True,
    initialization_method="estimated",
)

ajuste_holt = modelo_holt.fit(optimized=True)
pronostico_holt = ajuste_holt.forecast(h_prueba)

#3. HW Aditivo
# La estacionalidad aditiva representa aumentos o disminuciones de tamaño
# aproximadamente constante: por ejemplo, +30 unidades cada diciembre.
modelo_winters_aditivo = ExponentialSmoothing(
    entrenamiento,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=m,
    initialization_method="estimated",
)

ajuste_winters_aditivo = modelo_winters_aditivo.fit(optimized=True)
pronostico_winters_aditivo = ajuste_winters_aditivo.forecast(h_prueba)

#4. HW Multiplicativo
# La estacionalidad multiplicativa representa efectos porcentuales. Su uso
# exige observaciones positivas, condición que cumple el tipo de cambio.
modelo_holt_winters_multiplicativo = ExponentialSmoothing(
    entrenamiento,
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=m,
    initialization_method="estimated",
)

ajuste_holt_winters_multiplicativo = (
    modelo_holt_winters_multiplicativo.fit(optimized=True)
)
pronostico_holt_winters_multiplicativo = (
    ajuste_holt_winters_multiplicativo.forecast(h_prueba)
)

In [ ]:
# Comparación gráfica fuera de muestra

comparacion = pd.DataFrame(index=prueba.index)
comparacion["Real"] = prueba
comparacion["SES"] = np.asarray(pronostico_ses)
comparacion["Holt"] = np.asarray(pronostico_holt)
comparacion["Winters aditivo"] = np.asarray(pronostico_winters_aditivo)
comparacion["Holt-Winters multiplicativo"] = np.asarray(
    pronostico_holt_winters_multiplicativo
)

plt.figure(figsize=(14, 6))
plt.plot(
    entrenamiento.index[-24:],
    entrenamiento.iloc[-24:],
    color="#7A8790",
    label="Entrenamiento",
)
plt.plot(prueba.index, prueba, color="#123047", linewidth=2.8, label="Real")
plt.plot(comparacion.index, comparacion["SES"], label="SES")
plt.plot(comparacion.index, comparacion["Holt"], label="Holt")
plt.plot(
    comparacion.index,
    comparacion["Winters aditivo"],
    label="Winters aditivo",
)
plt.plot(
    comparacion.index,
    comparacion["Holt-Winters multiplicativo"],
    label="Holt-Winters multiplicativo",
)
plt.axvline(prueba.index[0], color="#C84A4A", linestyle="--", alpha=0.8)
plt.title("Comparación de pronósticos en los últimos 12 meses")
plt.xlabel("Fecha")
plt.ylabel("Moneda local por US$")
plt.legend(ncol=2)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
#Métricas de evaluación
# MAE: error absoluto medio, en las unidades originales.
# RMSE: penaliza con mayor fuerza los errores grandes.
# MAPE: error porcentual medio. No debe usarse si la serie contiene ceros.

error_ses = prueba.to_numpy() - comparacion["SES"].to_numpy()
error_holt = prueba.to_numpy() - comparacion["Holt"].to_numpy()
error_wa = prueba.to_numpy() - comparacion["Winters aditivo"].to_numpy()
error_hwm = (
    prueba.to_numpy()
    - comparacion["Holt-Winters multiplicativo"].to_numpy()
)

resultados = pd.DataFrame(
    {
        "MAE": [
            np.mean(np.abs(error_ses)),
            np.mean(np.abs(error_holt)),
            np.mean(np.abs(error_wa)),
            np.mean(np.abs(error_hwm)),
        ],
        "RMSE": [
            np.sqrt(np.mean(error_ses**2)),
            np.sqrt(np.mean(error_holt**2)),
            np.sqrt(np.mean(error_wa**2)),
            np.sqrt(np.mean(error_hwm**2)),
        ],
        "MAPE (%)": [
            100 * np.mean(np.abs(error_ses / prueba.to_numpy())),
            100 * np.mean(np.abs(error_holt / prueba.to_numpy())),
            100 * np.mean(np.abs(error_wa / prueba.to_numpy())),
            100 * np.mean(np.abs(error_hwm / prueba.to_numpy())),
        ],
    },
    index=[
        "SES",
        "Holt",
        "Winters aditivo",
        "Holt-Winters multiplicativo",
    ],
)

resultados = resultados.sort_values("RMSE")

print("\nMétricas fuera de muestra")
print(resultados.round(3))
print("\nMejor modelo según RMSE:", resultados.index[0])


In [ ]:
#Parámetros estimados
parametros_estimados = pd.DataFrame(
    {
        "SES": [
            ajuste_ses.params.get("smoothing_level", np.nan),
            np.nan,
            np.nan,
        ],
        "Holt": [
            ajuste_holt.params.get("smoothing_level", np.nan),
            ajuste_holt.params.get("smoothing_trend", np.nan),
            np.nan,
        ],
        "Winters aditivo": [
            ajuste_winters_aditivo.params.get("smoothing_level", np.nan),
            ajuste_winters_aditivo.params.get("smoothing_trend", np.nan),
            ajuste_winters_aditivo.params.get("smoothing_seasonal", np.nan),
        ],
        "Holt-Winters multiplicativo": [
            ajuste_holt_winters_multiplicativo.params.get(
                "smoothing_level", np.nan
            ),
            ajuste_holt_winters_multiplicativo.params.get(
                "smoothing_trend", np.nan
            ),
            ajuste_holt_winters_multiplicativo.params.get(
                "smoothing_seasonal", np.nan
            ),
        ],
    },
    index=["alpha: nivel", "beta: tendencia", "gamma: estacionalidad"],
)

print("\nParámetros de suavizamiento")
print(parametros_estimados.round(4))


In [ ]:
# Pronóstico final
h_futuro = 12

ses_final = SimpleExpSmoothing(
    serie,
    initialization_method="estimated",
).fit(optimized=True)

holt_final = Holt(
    serie,
    damped_trend=True,
    initialization_method="estimated",
).fit(optimized=True)

wa_final = ExponentialSmoothing(
    serie,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=m,
    initialization_method="estimated",
).fit(optimized=True)

hwm_final = ExponentialSmoothing(
    serie,
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=m,
    initialization_method="estimated",
).fit(optimized=True)

fecha_futura = pd.date_range(
    serie.index.max() + pd.offsets.MonthBegin(1),
    periods=h_futuro,
    freq="MS",
)

pronosticos_futuros = pd.DataFrame(index=fecha_futura)
pronosticos_futuros["SES"] = np.asarray(ses_final.forecast(h_futuro))
pronosticos_futuros["Holt"] = np.asarray(holt_final.forecast(h_futuro))
pronosticos_futuros["Winters aditivo"] = np.asarray(
    wa_final.forecast(h_futuro)
)
pronosticos_futuros["Holt-Winters multiplicativo"] = np.asarray(
    hwm_final.forecast(h_futuro)
)

mejor_modelo = resultados.index[0]

plt.figure(figsize=(14, 6))
plt.plot(serie.index[-60:], serie.iloc[-60:], color="#123047", label="Observado")
plt.plot(
    pronosticos_futuros.index,
    pronosticos_futuros[mejor_modelo],
    color="#E67E22",
    linewidth=2.5,
    label=f"Pronóstico: {mejor_modelo}",
)
plt.axvline(serie.index.max(), color="#C84A4A", linestyle="--")
plt.title("Pronóstico de 12 meses con el modelo seleccionado")
plt.xlabel("Fecha")
plt.ylabel("Moneda local por US$")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("\nPronósticos futuros")
print(pronosticos_futuros[[mejor_modelo]].round(2))


📝 **Apunte — Ejercicio 1 (h_prueba de 24 a 12)**

**Qué hizo la profe:** copió las celdas de separación, modelos, gráfico, métricas, parámetros y pronóstico final, cambiando **solo** `h_prueba = 12` (y en el gráfico mostró 24 meses de entrenamiento en vez de 48, y el título “últimos 12 meses”).

**Resultado:**

| | h_prueba = 24 | h_prueba = 12 |
|---|---|---|
| Ganador | Winters aditivo (RMSE 32,04) | **SES** (RMSE 59,07) |

**Respuesta: Sí, cambia el modelo ganador.**

**Cómo interpretarlo:**
- La elección del modelo **depende de la ventana de prueba**: con 12 meses se evalúa en un período distinto (los últimos 12, con movimientos que ningún modelo anticipó).
- Con **menos datos de prueba** la métrica es **menos confiable** (un par de meses raros pesan mucho).
- Los RMSE de los 4 modelos están muy cerca entre sí → ninguno es claramente superior. Coherente con un tipo de cambio que se mueve casi al azar.
- Como gana SES, el pronóstico final es una **línea plana** (917,01 todos los meses).

### 2. Ajuste Holt sin amortiguación: damped_trend=False. Compare el RMSE.

In [ ]:
# Se reservan los últimos 24 meses para una evaluación fuera de muestra.
h_prueba = 24
entrenamiento = serie.iloc[:-h_prueba]
prueba = serie.iloc[-h_prueba:]

print("\nTamaño de entrenamiento:", len(entrenamiento))
print("Tamaño de prueba:", len(prueba))

In [ ]:
# Métodos

#1. SES
# Adecuado para una serie que fluctúa alrededor de un nivel relativamente
# estable. No incorpora tendencia ni estacionalidad.
modelo_ses = SimpleExpSmoothing(
    entrenamiento,
    initialization_method="estimated",
)

ajuste_ses = modelo_ses.fit(optimized=True)
pronostico_ses = ajuste_ses.forecast(h_prueba)

#2. Holt
# Incorpora una tendencia. Se utiliza tendencia amortiguada para evitar que
# una pendiente estimada se prolongue indefinidamente con igual intensidad.
modelo_holt = Holt(
    entrenamiento,
    damped_trend=False,
    initialization_method="estimated",
)

ajuste_holt = modelo_holt.fit(optimized=True)
pronostico_holt = ajuste_holt.forecast(h_prueba)

#3. HW Aditivo
# La estacionalidad aditiva representa aumentos o disminuciones de tamaño
# aproximadamente constante: por ejemplo, +30 unidades cada diciembre.
modelo_winters_aditivo = ExponentialSmoothing(
    entrenamiento,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=m,
    initialization_method="estimated",
)

ajuste_winters_aditivo = modelo_winters_aditivo.fit(optimized=True)
pronostico_winters_aditivo = ajuste_winters_aditivo.forecast(h_prueba)

#4. HW Multiplicativo
# La estacionalidad multiplicativa representa efectos porcentuales. Su uso
# exige observaciones positivas, condición que cumple el tipo de cambio.
modelo_holt_winters_multiplicativo = ExponentialSmoothing(
    entrenamiento,
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=m,
    initialization_method="estimated",
)

ajuste_holt_winters_multiplicativo = (
    modelo_holt_winters_multiplicativo.fit(optimized=True)
)
pronostico_holt_winters_multiplicativo = (
    ajuste_holt_winters_multiplicativo.forecast(h_prueba)
)

In [ ]:
# Comparación gráfica fuera de muestra

comparacion = pd.DataFrame(index=prueba.index)
comparacion["Real"] = prueba
comparacion["SES"] = np.asarray(pronostico_ses)
comparacion["Holt"] = np.asarray(pronostico_holt)
comparacion["Winters aditivo"] = np.asarray(pronostico_winters_aditivo)
comparacion["Holt-Winters multiplicativo"] = np.asarray(
    pronostico_holt_winters_multiplicativo
)

plt.figure(figsize=(14, 6))
plt.plot(
    entrenamiento.index[-24:],
    entrenamiento.iloc[-24:],
    color="#7A8790",
    label="Entrenamiento",
)
plt.plot(prueba.index, prueba, color="#123047", linewidth=2.8, label="Real")
plt.plot(comparacion.index, comparacion["SES"], label="SES")
plt.plot(comparacion.index, comparacion["Holt"], label="Holt")
plt.plot(
    comparacion.index,
    comparacion["Winters aditivo"],
    label="Winters aditivo",
)
plt.plot(
    comparacion.index,
    comparacion["Holt-Winters multiplicativo"],
    label="Holt-Winters multiplicativo",
)
plt.axvline(prueba.index[0], color="#C84A4A", linestyle="--", alpha=0.8)
plt.title("Comparación de pronósticos en los últimos 12 meses")
plt.xlabel("Fecha")
plt.ylabel("Moneda local por US$")
plt.legend(ncol=2)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
#Métricas de evaluación
# MAE: error absoluto medio, en las unidades originales.
# RMSE: penaliza con mayor fuerza los errores grandes.
# MAPE: error porcentual medio. No debe usarse si la serie contiene ceros.

error_ses = prueba.to_numpy() - comparacion["SES"].to_numpy()
error_holt = prueba.to_numpy() - comparacion["Holt"].to_numpy()
error_wa = prueba.to_numpy() - comparacion["Winters aditivo"].to_numpy()
error_hwm = (
    prueba.to_numpy()
    - comparacion["Holt-Winters multiplicativo"].to_numpy()
)

resultados = pd.DataFrame(
    {
        "MAE": [
            np.mean(np.abs(error_ses)),
            np.mean(np.abs(error_holt)),
            np.mean(np.abs(error_wa)),
            np.mean(np.abs(error_hwm)),
        ],
        "RMSE": [
            np.sqrt(np.mean(error_ses**2)),
            np.sqrt(np.mean(error_holt**2)),
            np.sqrt(np.mean(error_wa**2)),
            np.sqrt(np.mean(error_hwm**2)),
        ],
        "MAPE (%)": [
            100 * np.mean(np.abs(error_ses / prueba.to_numpy())),
            100 * np.mean(np.abs(error_holt / prueba.to_numpy())),
            100 * np.mean(np.abs(error_wa / prueba.to_numpy())),
            100 * np.mean(np.abs(error_hwm / prueba.to_numpy())),
        ],
    },
    index=[
        "SES",
        "Holt",
        "Winters aditivo",
        "Holt-Winters multiplicativo",
    ],
)

resultados = resultados.sort_values("RMSE")

print("\nMétricas fuera de muestra")
print(resultados.round(3))
print("\nMejor modelo según RMSE:", resultados.index[0])


📝 **Apunte — Ejercicio 2 (Holt sin amortiguar)**

**Qué hizo la profe:** volvió a `h_prueba = 24` y en el modelo de Holt cambió `damped_trend=True` → **`damped_trend=False`** (solo en Holt; los demás quedan igual).

⚠️ Detalle: el título del gráfico dice “últimos 12 meses”, pero aquí la prueba es de **24** (quedó del ejercicio anterior). Si lo copias, corrige el título.

| Holt | RMSE |
|---|---|
| Amortiguado (`damped_trend=True`) | 32,89 |
| **Sin amortiguar** (`damped_trend=False`) | **41,90** |

**Interpretación:** sin amortiguar, Holt **prolonga la pendiente con la misma fuerza** todos los meses → el error crece a medida que avanza el horizonte y queda **último** del ranking. La amortiguación evita extrapolar una tendencia indefinidamente, y por eso es más prudente en series como el tipo de cambio.

### 3. Compare la amplitud estacional en unidades y como porcentaje del nivel.

In [ ]:
#Forma 1
from statsmodels.tsa.seasonal import STL
import pandas as pd
import numpy as np


# Ajuste STL sobre el conjunto de entrenamiento
stl = STL(entrenamiento, period=12, robust=True).fit()

nivel = stl.trend
estacionalidad = stl.seasonal

# ---------------------------------------------------------
# 1. Amplitud estacional en unidades
# ---------------------------------------------------------

amp_unidades = estacionalidad.max() - estacionalidad.min()
print("Amplitud estacional (unidades):", round(amp_unidades, 4))

# ---------------------------------------------------------
# 2. Amplitud estacional como porcentaje del nivel
# ---------------------------------------------------------

nivel_promedio = nivel.mean()
amp_porcentaje = (amp_unidades / nivel_promedio) * 100

print("Amplitud estacional (% del nivel):", round(amp_porcentaje, 4), "%")

# ---------------------------------------------------------
# 3. Tabla resumen
# ---------------------------------------------------------

tabla = pd.DataFrame({
    "Amplitud (unidades)": [amp_unidades],
    "Nivel promedio": [nivel_promedio],
    "Amplitud (%)": [amp_porcentaje]
})

print("\nResumen:")
print(tabla)


📝 **Apunte — Ejercicio 3, Forma 1 (STL)**

- **STL** (*Seasonal-Trend decomposition using LOESS*) es otra descomposición, más flexible que la clásica: **permite que la estacionalidad cambie con los años**. `robust=True` hace que los atípicos influyan menos.
- `stl.trend` = nivel; `stl.seasonal` = estacionalidad.
- **Amplitud en unidades** = `max − min` de la estacionalidad (cuánto varía de su punto más alto al más bajo).
- **Amplitud en %** = amplitud ÷ nivel promedio × 100.

**Resultado:** amplitud ≈ **99,24 pesos** ≈ **16,63%** del nivel promedio (596,57). Ojo: como STL deja que la estacionalidad cambie, este max − min toma el pico y el valle de **todos los años** del entrenamiento, por eso es más grande que en la Forma 2.

In [ ]:
#Forma 2
from statsmodels.tsa.seasonal import seasonal_decompose
import pandas as pd

# Descomposición clásica
descomposicion_aditiva = seasonal_decompose(
    serie,
    model='additive',
    period=12
)

descomposicion_multiplicativa = seasonal_decompose(
    serie,
    model='multiplicative',
    period=12
)

# Tabla con los primeros 12 meses del patrón estacional
estacionalidad_promedio = pd.DataFrame({
    'Mes': ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'],
    'Efecto_estacional_aditivo': descomposicion_aditiva.seasonal.iloc[:12].values,
    'Factor_estacional_multiplicativo': descomposicion_multiplicativa.seasonal.iloc[:12].values
})

print(estacionalidad_promedio.round(3))


In [ ]:
# Amplitud aditiva en unidades
amp_aditiva = (
    estacionalidad_promedio['Efecto_estacional_aditivo'].max()
    - estacionalidad_promedio['Efecto_estacional_aditivo'].min()
)

# Amplitud multiplicativa como porcentaje
amp_multiplicativa_pct = (
    estacionalidad_promedio['Factor_estacional_multiplicativo'].max()
    - estacionalidad_promedio['Factor_estacional_multiplicativo'].min()
) * 100

print("Amplitud estacional aditiva (unidades):", round(amp_aditiva, 4))
print("Amplitud estacional multiplicativa (%):", round(amp_multiplicativa_pct, 4), "%")


📝 **Apunte — Ejercicio 3, Forma 2 (descomposición clásica de la Clase 2)**

Es **el mismo código de la Clase 2** (`seasonal_decompose` + tabla de 12 meses), aplicado ahora a la `serie` del tipo de cambio.

- Efectos aditivos entre **−8,72 (abril)** y **+10,66 (octubre)** → amplitud **19,37 pesos**.
- Factores multiplicativos entre **0,985 y 1,015** → amplitud **≈ 2,96%**.
- Para la amplitud multiplicativa en %: (factor máx − factor mín) × 100.

**Lectura:** con el método clásico (un patrón promedio fijo), la estacionalidad del dólar es **pequeña**: ±1,5% del nivel.

In [ ]:
#Código para ver si la amplitud cambia con el nivel



stl = STL(serie, period=12, robust=True).fit()
est = stl.seasonal
nivel = stl.trend

# Agrupar por año para ver amplitud año a año
df = pd.DataFrame({
    "est": est,
    "nivel": nivel
})
df["año"] = df.index.year

tabla = df.groupby("año").agg(
    amp_unidades=("est", lambda x: x.max() - x.min()),
    nivel_prom=("nivel", "mean")
)

tabla["amp_pct"] = tabla["amp_unidades"] / tabla["nivel_prom"] * 100

print(tabla.round(3))


📝 **Apunte — tabla año a año**

- Hace STL sobre **toda** la serie y arma un DataFrame con estacionalidad y nivel.
- `df.index.year` saca el año; `groupby("año").agg(...)` calcula **por año**: la amplitud (`max − min` con una `lambda`) y el nivel promedio.
- `amp_pct` = amplitud ÷ nivel × 100.

📊 **Cómo leer la tabla:** compara **cómo se mueven juntas** `amp_unidades` y `nivel_prom`. En los años de nivel más bajo (2012–2016, nivel ~485–670) la amplitud es ~11–19 pesos; en los años de nivel alto (2020–2026, nivel ~760–930) la amplitud es ~68–86 pesos. **La amplitud en unidades crece cuando el nivel crece** → ese es el argumento central del ejercicio 4.

### 4. Justifique si corresponde estacionalidad aditiva o multiplicativa.

In [ ]:
#La serie presenta estacionalidad multiplicativa, y esto se justifica por:

#1. Evidencia desde la descomposición clásica (seasonal_decompose) mostró:

#Efectos aditivos entre aproximadamente −9 y +11 → amplitud ≈ 19 unidades.

#Factores multiplicativos entre 0.985 y 1.015 → amplitud ≈ 3%.

# Esto indica que:

# El patrón estacional como porcentaje del nivel es pequeño y estable.

# El patrón estacional en unidades es pequeño porque la descomposición clásica entrega solo el promedio del ciclo.

#2. Evidencia desde STL (estacionalidad real en toda la serie) STL entregó:

#Amplitud estacional en unidades: 99.24

#Amplitud estacional como porcentaje del nivel: 16.63%

#Por lo tanto,
#La amplitud en unidades es grande (≈ 100)
#crece cuando el nivel de la serie crece.

#La amplitud porcentual es estable (≈ 16%)
#la estacionalidad es proporcional al nivel.

#La amplitud en unidades no es constante en el tiempo, por lo tanto, es mejor la descomposición multiplicativa.

📝 **Apunte — Ejercicio 4: cómo justificar**

La profe concluye **multiplicativa** con este razonamiento (memorízalo como estructura de respuesta):
1. **Evidencia numérica** de la descomposición clásica (amplitud en unidades vs en %).
2. **Evidencia STL**: la amplitud en unidades es grande y **no es constante en el tiempo**; **crece cuando el nivel crece**.
3. **Conclusión:** si la estacionalidad crece con el nivel → es proporcional al nivel → **multiplicativa**. (Si la amplitud en unidades fuera estable → aditiva.)

💡 **Para que no te pillen:** en el ejercicio 1 y en la evaluación original **el RMSE favoreció al aditivo**. ¿Contradicción? No necesariamente:
- La **justificación** se refiere a la **forma** de la estacionalidad (cómo cambia su tamaño).
- El **RMSE** mide **qué tan bien pronostica** en un período concreto.
- Además la estacionalidad del dólar es **pequeña** (≈ 3%) y **γ ≈ 0**, así que ambos modelos pronostican casi lo mismo y la diferencia en RMSE es poca.

Si te preguntan en el control, lo ideal es **argumentar con los números que te dé el código** (las amplitudes y la tabla año a año).

### 5. Grafique los residuos del modelo ganador y calcule su FAC.

In [ ]:
#Crear la serie de residuos y su “crecimiento”

hw_ganador = ajuste_winters_aditivo   # este sí tiene .resid


# Residuos del modelo ganador
residuos = hw_ganador.resid.dropna()


# "Crecimiento" de los residuos: diferencia porcentual
residuos_crecimiento = residuos.pct_change().dropna()


#FAC y FACP
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

max_lags = min(15, len(residuos_crecimiento) // 2 - 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# FAC del nivel (residuos)
plot_acf(residuos, lags=max_lags, ax=axes[0, 0], zero=False)
axes[0, 0].set_title('FAC: residuos del modelo ganador')

# FACP del nivel (residuos)
plot_pacf(
    residuos,
    lags=max_lags,
    ax=axes[0, 1],
    zero=False,
    method='ywm',
)
axes[0, 1].set_title('FACP: residuos del modelo ganador')

# FAC del crecimiento de los residuos
plot_acf(residuos_crecimiento, lags=max_lags, ax=axes[1, 0], zero=False)
axes[1, 0].set_title('FAC: crecimiento de los residuos')

# FACP del crecimiento de los residuos
plot_pacf(
    residuos_crecimiento,
    lags=max_lags,
    ax=axes[1, 1],
    zero=False,
    method='ywm',
)
axes[1, 1].set_title('FACP: crecimiento de los residuos')

plt.tight_layout()
plt.show()


📝 **Apunte — Ejercicio 5: residuos y FAC**

- **Residuo** = real − ajustado, **dentro del entrenamiento**. `hw_ganador.resid` los entrega; `.dropna()` bota vacíos.
- ⚠️ **Ojo muy importante:** la profe escribió **a mano** `hw_ganador = ajuste_winters_aditivo` porque ese fue el ganador. **Si en tu control gana otro, cambia esa línea:**

| Ganador | Escribe |
|---|---|
| SES | `hw_ganador = ajuste_ses` |
| Holt | `hw_ganador = ajuste_holt` |
| Winters aditivo | `hw_ganador = ajuste_winters_aditivo` |
| Holt-Winters multiplicativo | `hw_ganador = ajuste_holt_winters_multiplicativo` |

- `residuos.pct_change()` = variación porcentual de los residuos (lo que la profe llama “crecimiento”).
- `max_lags = min(15, len(...) // 2 - 1)` → se miran hasta 15 rezagos (lags).
- `plt.subplots(2, 2)` → una grilla de **4 gráficos**: arriba FAC y FACP de los residuos; abajo FAC y FACP del crecimiento.
- `zero=False` no dibuja el rezago 0 (que siempre vale 1 y no aporta). `method='ywm'` es el método de cálculo de la FACP.

📊 **Cómo leer una FAC (función de autocorrelación)**

- Eje X = **rezago** (lag): 1 = correlación del residuo con el del mes anterior, 2 = con el de hace 2 meses, etc.
- Cada **barra/palito** = cuánta correlación hay en ese rezago (entre −1 y 1).
- La **zona sombreada azul** = banda de confianza del 95%.
  - **Barras dentro de la banda** → no hay autocorrelación significativa → **los residuos son ruido** → el modelo capturó bien la estructura ✔️
  - **Barras que se salen** → queda **patrón sin explicar** (por ejemplo, un rezago 12 fuera de la banda = estacionalidad no capturada) ✖️
- La **FACP** (parcial) mide la correlación con el rezago *k* **quitando** el efecto de los rezagos intermedios.

**Para responder:** lo principal es la fila de arriba (residuos). Di cuántos rezagos se salen de la banda y en cuáles; si casi todos están dentro, los residuos se comportan como ruido blanco.

### Interpretación

In [ ]:
#¿Qué información ignora SES que sí incorpora Holt?

# SES ignora la tendencia, supone que la serie no crece ni decrece de manera sistemática, solo actualiza un nivel:
# Por lo tanto, no tiene pendiente, ni crecimiento. No incluye ni aceleración o desaceleración de la serie, si hay crecimiento
#o decrecimiento a largo plazo.
#Holt sí incorpora tendencia, puede capturar crecimiento, capturar descenso y proyectar una trayectoria lineal a diferencia de SES.
#Es un modelo de nivel+tendencia, incluye shocks persistentes, ajusta cambios de dirección de la serie, por lo que los pronósticos
#serán más realistas.



In [ ]:
#¿Qué representa gamma en Holt-Winters?
#Gamma (γ) en Holt‑Winters es el parámetro de suavizamiento de la estacionalidad.
#Es decir, controla qué tan rápido se actualiza el componente estacional cuando llega un nuevo dato.

In [ ]:
#¿Por qué el modelo multiplicativo exige valores positivos?

#Porque en Holt‑Winters multiplicativo la estacionalidad y los pronósticos se construyen multiplicando.
#El modelo multiplicativo exige valores positivos porque la estacionalidad se expresa como un factor proporcional. 
#Los factores multiplicativos no pueden ser negativos, y las ecuaciones del modelo usan ratios que solo tienen sentido 
#con valores positivos.


In [ ]:
#¿Por qué la evaluación debe hacerse fuera de muestra?

#La evaluación debe hacerse fuera de muestra porque solo así podemos medir la capacidad real de pronóstico del modelo. 
#Dentro de muestra el modelo ve los datos que ya conoce y el error es engañosamente bajo; fuera de muestra revela si el 
#modelo generaliza y evita el sobreajuste.

In [ ]:
#¿El menor RMSE garantiza residuos sin autocorrelación? Explique.
#No. El menor RMSE no garantiza ausencia de autocorrelación en los residuos. 
#RMSE evalúa precisión predictiva fuera de muestra, mientras que la autocorrelación evalúa si el modelo capturó 
#toda la estructura dentro de muestra. Un modelo puede predecir bien y aun así dejar patrones sin modelar en los residuos.

📝 **Apunte — Preguntas de interpretación (versión corta para memorizar)**

| Pregunta | Idea clave |
|---|---|
| ¿Qué ignora SES que sí incorpora Holt? | La **tendencia (pendiente)**. SES = solo nivel → pronóstico plano. Holt = nivel + pendiente → pronóstico con dirección. |
| ¿Qué representa γ? | El parámetro de suavizamiento de la **estacionalidad**: qué tan rápido se actualiza el patrón estacional. |
| ¿Por qué el multiplicativo exige positivos? | Porque la estacionalidad es un **factor** (se multiplica / se divide). Con ceros o negativos los factores y las razones no tienen sentido. |
| ¿Por qué fuera de muestra? | Porque dentro de muestra el modelo ya vio los datos y el error es engañosamente bajo; fuera de muestra se mide la **capacidad real de pronóstico** y se evita el sobreajuste. |
| ¿Menor RMSE = residuos sin autocorrelación? | **No.** RMSE mide precisión; la FAC de residuos mide si quedó estructura. Son cosas distintas: se revisan las dos. |

📝 **Ejercicio 6 (no resuelto en el notebook): repetir para otro país**

Copia la celda de descarga y cambia **`CHL`** por otro código (`PER`, `COL`, `MEX`, `BRA`…) en `url_gem`. Después vuelve a ejecutar **todas** las celdas siguientes en orden (gráfico, separación, modelos, comparación, métricas, parámetros, pronóstico, residuos). Cambia los títulos que dicen “Chile”. Compara: ¿gana el mismo modelo? ¿los α también son ≈ 1?

---
# 🧾 CHULETA PARA EL CONTROL

### 1. “Me piden X” → “copio esto”

| Me piden… | Archivo de la profe | Celda(s) |
|---|---|---|
| Crear una serie simulada / graficarla | Transformaciones | creación de `df` + gráfico “Serie original” |
| Logaritmo / Box-Cox / Raíz / Yeo-Johnson / Z / Winsorizar | Transformaciones | la celda de cada sección (1 a 6) |
| Comparar visualmente las transformaciones | Transformaciones | “Comparación visual” (7 subgráficos) |
| Descomponer una serie (aditiva / multiplicativa) | Descomposición | secciones 2, 4, 5, 7 |
| Tabla de efectos / factores estacionales por mes | Descomposición | sección 9 |
| Descargar datos del Banco Mundial | Holt-Winters | sección 1 |
| Separar entrenamiento / prueba | Holt-Winters | sección 2 |
| Ajustar SES, Holt, HW aditivo, HW multiplicativo | Holt-Winters | secciones 3 a 6 (o la celda “Métodos” de la práctica, que tiene los 4 juntos) |
| Gráfico de comparación fuera de muestra | Holt-Winters | sección 7 |
| MAE, RMSE, MAPE y ganador | Holt-Winters | sección 8 |
| α, β, γ estimados | Holt-Winters | sección 9 |
| Pronóstico futuro | Holt-Winters | sección 10 |
| Amplitud estacional (unidades y %) | Práctica | ejercicio 3 (Forma 1 STL, Forma 2 clásica, tabla año a año) |
| Residuos y FAC/FACP | Práctica | ejercicio 5 |

### 2. Números “escritos a mano” que debes revisar si cambian los datos

| Dónde | Qué está fijo | Qué hacer |
|---|---|---|
| Clase 1, Yeo-Johnson con estilo | `lambda_yj = 0.6552` | poner el λ nuevo |
| Clase 1, celda de gráficos | `lmbda=0.6575`, `lmbda=0.6552` | poner los λ nuevos |
| Clase 1, creación de datos | `periods=60` y `np.random.normal(0, 2, 60)` | deben ser iguales |
| Clase 1, atípicos | fechas `'2021-11-01'`, `'2023-11-01'` | deben estar dentro del rango |
| Clase 2 | `periods=48` = cantidad de números en `ventas` | deben coincidir |
| Clase 2 y práctica | `period=12`, `m = 12`, `.iloc[:12]` y meses `'Ene'…'Dic'` | cambiar si no es mensual o no parte en enero |
| Clase 3 | `CHL`, `DPANUSLCU`, `"1995M01:2026M12"` | país, indicador, fechas |
| Clase 3 | `h_prueba = 24`, `h_futuro = 12` | horizonte de prueba y de pronóstico |
| Clase 3, gráfico de comparación | `[-48:]` y el título “últimos 24 meses” | ajustar títulos al nuevo `h_prueba` |
| Práctica ej. 5 | `hw_ganador = ajuste_winters_aditivo` | poner el ajuste del modelo que **ganó** |

### 3. Errores típicos

| Error | Causa probable | Solución |
|---|---|---|
| `NameError: name '...' is not defined` | No ejecutaste una celda anterior | Ejecutar desde arriba (*Run All*) |
| `ModuleNotFoundError: No module named 'scipy'` (u otro) | Falta instalar un paquete | `%pip install scipy` y reiniciar kernel |
| `AttributeError: ... requires jinja2` | Falta jinja2 para `df.style` | `%pip install jinja2` |
| `ValueError: Length of values ... does not match` | Distinto número de datos que de fechas | Igualar `periods` con la cantidad de datos |
| `ValueError` en Box-Cox | Hay ceros o negativos | Usar Yeo-Johnson |
| Error en `seasonal="mul"` | Hay ceros o negativos | Usar `"add"` |
| `ValueError` en `seasonal_decompose` | Menos de 2 ciclos completos | Más datos o `period` correcto |
| `RuntimeError: La API no devolvió observaciones` | Sin internet o país/indicador sin datos | Revisar conexión / probar otro país |
| `RuntimeError: Persisten brechas mayores a dos meses` | La serie tiene huecos largos | Es a propósito: elegir otro período o país |

### 4. Plantilla para responder preguntas de interpretación

1. **Qué hice** (qué celda/modelo y con qué parámetros).
2. **Qué salió** (el número: RMSE, λ, amplitud, factor…).
3. **Qué significa** en palabras (en las unidades de la serie o en %).
4. **Conclusión** (qué modelo/transformación conviene y por qué).

*Ejemplo:* “Con h_prueba = 24, el menor RMSE lo obtuvo Winters aditivo (32,04), seguido muy de cerca por SES (32,56). En promedio el modelo se equivoca ~25,5 pesos por dólar (MAPE 2,7%). La diferencia es mínima, lo que sugiere que la tendencia y la estacionalidad aportan poca información adicional: con α ≈ 1, el tipo de cambio se comporta casi como un paseo aleatorio.”

### 🅱️ Plan B — si en el control te entregan un archivo en vez de datos simulados o de la API

> ⚠️ Esto **no está en las clases de la profe**; lo dejo solo como referencia por si te pasan un Excel/CSV. Pregúntale a la profe si está permitido o si los datos vendrán dentro del código.

La idea es **construir el mismo objeto que usa la profe** (una serie con fechas como índice) y desde ahí usar sus celdas sin cambios:

```python
datos = pd.read_excel("archivo.xlsx")          # o pd.read_csv("archivo.csv")
datos["fecha"] = pd.to_datetime(datos["fecha"])  # el nombre de la columna puede ser otro
serie = datos.set_index("fecha")["valor"].asfreq("MS")
```

Con `serie` lista, sigues con la Clase 2 (`seasonal_decompose(serie, ...)`) o con la Clase 3 desde “Exploración y separación temporal”.

---
✅ **Autoevaluación final — Clase 3 y práctica**

<details><summary>1. En un control, con h_prueba = 18, el ranking quedó: Holt 20,1 · SES 20,3 · HW aditivo 22,0 · HW multiplicativo 23,5 (RMSE). ¿Qué modelo eliges y qué cambias en el ejercicio de residuos?</summary>

Elijo **Holt** (menor RMSE). En el ejercicio 5 cambio `hw_ganador = ajuste_holt`. Comento que la diferencia con SES es muy pequeña (0,2).
</details>

<details><summary>2. α = 0,15 en SES. ¿Qué significa?</summary>

El nivel da **poco peso** al dato nuevo (15%) y mucho al pasado (85%): el modelo es **suave**, con mucha memoria; reacciona lento a los cambios.
</details>

<details><summary>3. ¿Por qué el pronóstico de SES es una línea plana?</summary>

Porque $\widehat y_{t+h\mid t}=\ell_t$: el pronóstico para cualquier horizonte es el último nivel estimado. No tiene pendiente ni estacionalidad.
</details>

<details><summary>4. MAE = 20 y RMSE = 35. ¿Qué te dice la diferencia?</summary>

Como el RMSE castiga más los errores grandes, que sea bastante mayor que el MAE indica que hubo **algunos errores grandes** (meses donde el modelo falló mucho).
</details>

<details><summary>5. ¿Por qué no se usa MAPE si la serie tiene ceros?</summary>

Porque MAPE divide por el valor real: dividir por 0 no está definido (y por valores cercanos a 0 da porcentajes enormes).
</details>

<details><summary>6. En la FAC de los residuos, la barra del rezago 12 se sale de la banda azul. ¿Qué indica?</summary>

Que queda **autocorrelación en el rezago 12** → probablemente hay **estacionalidad anual que el modelo no capturó**.
</details>

<details><summary>7. ¿Para qué se vuelve a estimar el modelo con toda la serie al final?</summary>

La prueba sirve para **elegir** el modelo; una vez elegido, se estima con **todos** los datos (incluidos los más recientes) para que el pronóstico futuro use toda la información disponible.
</details>

<details><summary>8. ¿Qué diferencia hay entre la descomposición clásica y STL en el ejercicio 3?</summary>

La clásica entrega **un patrón estacional fijo** (promedio, igual todos los años); STL permite que **la estacionalidad cambie en el tiempo** y es robusta a atípicos. Por eso la amplitud con STL (≈ 99) es mayor que con la clásica (≈ 19).
</details>

**¡Mucho éxito en el control! 💪**